In [ ]:
xls = pd.ExcelFile(r"C:\Users\malik\AppData\Local\Packages\5319275A.WhatsAppDesktop_cv1g1gvanyjgm\LocalState\sessions\711B537ADBC4AEA5082BDAB60FD84D562764D598\transfers\2026-25\LAC_CELL_CIRCLE_2G_SITES_DATA_FINAL.xlsx")
print(xls.sheet_names)

In [ ]:
file_path = r"C:\Users\malik\AppData\Local\Packages\5319275A.WhatsAppDesktop_cv1g1gvanyjgm\LocalState\sessions\711B537ADBC4AEA5082BDAB60FD84D562764D598\transfers\2026-25\LAC_CELL_CIRCLE_2G_SITES_DATA_FINAL.xlsx"

In [ ]:
df1 = pd.read_excel(file_path,sheet_name=0)
df2 = pd.read_excel(file_path,sheet_name=1)
df3 = pd.read_excel(file_path,sheet_name=2)

In [ ]:
df1["Period"] = "May 8_10"
df2["Period"] = "May 15_17"
df3["Period"] = "May 22_24"

In [ ]:
master_df = pd.concat([df1, df2, df3], ignore_index=True)
master_df.head()

In [ ]:
master_df.shape

In [ ]:
master_df.columns = (
    master_df.columns
    .str.strip()
    .str.upper()
    .str.replace(" ", "_")
)
master_df.columns

In [ ]:
total_sims = master_df["SIM_COUNT"].sum()
total_sessions = master_df["SESSION_COUNT"].sum()
total_cells = master_df["CELL_ID"].nunique()
active_cells = master_df[
    master_df["SIM_COUNT"] > 0
]["CELL_ID"].nunique()

In [ ]:
summary_metrics = pd.DataFrame({
    "Metric": [
        "Total SIMs",
        "Total Sessions",
        "Total Cells",
        "Active Cells"
    ],
    "Value": [
        total_sims,
        total_sessions,
        total_cells,
        active_cells
    ]
})

summary_metrics

In [ ]:
summary_metrics.to_csv(
    "summary_metrics.csv",
    index=False
)

In [ ]:
master_df.to_csv(
    "master_dataset.csv",
    index=False
)

In [ ]:
master_df.head()
master_df.columns
master_df.shape

In [ ]:
master_df.head()

In [ ]:
master_df.columns

In [ ]:
master_df.info()

In [ ]:
master_df.describe()

In [ ]:
total_sims = master_df["SIM_COUNT"].sum()

total_sessions = master_df["SESSION_COUNT"].sum()

total_cells = master_df["CELL_ID"].nunique()

active_cells = master_df[master_df["SIM_COUNT"] > 0]["CELL_ID"].nunique()

print("Total SIMs:", total_sims)
print("Total Sessions:", total_sessions)
print("Total Cells:", total_cells)
print("Active Cells:", active_cells)

In [ ]:
master_df.columns

In [ ]:
cell_pivot = master_df.pivot_table(
    index=["CELL_ID", "LAC_ID"],
    columns="PERIOD",
    values=["SIM_COUNT", "SESSION_COUNT"],
    aggfunc="sum"
)

cell_pivot.head()

In [ ]:
cell_pivot.columns = [
    f"{col[0]}_{col[1]}"
    for col in cell_pivot.columns
]

cell_pivot = cell_pivot.reset_index()

cell_pivot.head()

In [ ]:
cell_pivot.columns

In [ ]:
cell_pivot["SIM_CHANGE_PCT"] = (
    (
        cell_pivot["SIM_COUNT_May 22_24"]
        - cell_pivot["SIM_COUNT_May 8_10"]
    )
    / cell_pivot["SIM_COUNT_May 8_10"]
) * 100

In [ ]:
cell_pivot["SESSION_CHANGE_PCT"] = (
    (
        cell_pivot["SESSION_COUNT_May 22_24"]
        - cell_pivot["SESSION_COUNT_May 8_10"]
    )
    / cell_pivot["SESSION_COUNT_May 8_10"]
) * 100

In [ ]:
cell_pivot[
    [
        "CELL_ID",
        "SIM_CHANGE_PCT",
        "SESSION_CHANGE_PCT"
    ]
].head()

In [ ]:
def get_status(row):

    if (
        row["SIM_CHANGE_PCT"] <= -70
        and
        row["SESSION_CHANGE_PCT"] <= -70
    ):
        return "Critical"

    elif (
        row["SIM_CHANGE_PCT"] <= -40
        or
        row["SESSION_CHANGE_PCT"] <= -40
    ):
        return "Warning"

    else:
        return "Healthy"

In [ ]:
cell_pivot["STATUS"] = cell_pivot.apply(
    get_status,
    axis=1
)

In [ ]:
cell_pivot["STATUS"].value_counts()

In [ ]:
cell_status = cell_pivot[
    [
        "CELL_ID",
        "LAC_ID",
        "SIM_CHANGE_PCT",
        "SESSION_CHANGE_PCT",
        "STATUS"
    ]
]

cell_status.head()

In [ ]:
cell_status.to_csv(
    "cell_status.csv",
    index=False
)

In [ ]:
cell_pivot["STATUS"].value_counts()

In [ ]:
critical_cells = (
    cell_status["STATUS"] == "Critical"
).sum()

warning_cells = (
    cell_status["STATUS"] == "Warning"
).sum()

healthy_cells = (
    cell_status["STATUS"] == "Healthy"
).sum()

print("Critical:", critical_cells)
print("Warning:", warning_cells)
print("Healthy:", healthy_cells)

In [ ]:
worst_cell = cell_status.loc[
    cell_status["SIM_CHANGE_PCT"].idxmin()
]

worst_cell

In [ ]:
business_insights = pd.DataFrame({
    "INSIGHT":[
        "Critical Cells",
        "Warning Cells",
        "Healthy Cells"
    ],
    "VALUE":[
        critical_cells,
        warning_cells,
        healthy_cells
    ]
})

business_insights

In [ ]:
business_insights.to_csv(
    "business_insights.csv",
    index=False
)

In [ ]:
import numpy as np

np.random.seed(42)

unique_cells = cell_pivot[["CELL_ID","LAC_ID"]].copy()

unique_cells["LAT"] = np.random.uniform(
    28.40,
    28.90,
    len(unique_cells)
)

unique_cells["LONG"] = np.random.uniform(
    76.90,
    77.40,
    len(unique_cells)
)

unique_cells.head()

In [ ]:
unique_cells.to_csv(
    "cell_coordinates.csv",
    index=False
)

In [ ]:
unique_cells.shape

In [ ]:
unique_cells.shape

In [ ]:
unique_cells.head()

In [ ]:
from sklearn.neighbors import NearestNeighbors
import pandas as pd

# Use coordinates
coords = unique_cells[["LAT", "LONG"]]

# Model
nn = NearestNeighbors(n_neighbors=6, algorithm='ball_tree')
nn.fit(coords)

distances, indices = nn.kneighbors(coords)

# Build neighbor dataset
neighbor_data = []

for i in range(len(unique_cells)):
    for j in range(1, 6):  # skip itself (j=0)
        neighbor_data.append({
            "CELL_ID": unique_cells.iloc[i]["CELL_ID"],
            "NEIGHBOR_CELL": unique_cells.iloc[indices[i][j]]["CELL_ID"],
            "DISTANCE": distances[i][j]
        })

neighbor_df = pd.DataFrame(neighbor_data)

neighbor_df.head()

In [ ]:
neighbor_df.to_csv("neighbor_analysis.csv", index=False)

In [ ]:
neighbor_df.shape

In [ ]:
neighbor_df.shape

In [ ]:
neighbor_df.head()

In [ ]:
master_df.columns

In [ ]:
master_df["STATUS"] = master_df["SIM_COUNT"].apply(
    lambda x: "Critical" if x < 50 else "Healthy"
)

In [ ]:
status_df = master_df[["CELL_ID", "STATUS"]]

In [ ]:
merged = neighbor_df.merge(
    status_df,
    on="CELL_ID",
    how="left"
)

merged = merged.rename(columns={"STATUS": "CELL_STATUS"})

In [ ]:
merged = merged.merge(
    status_df,
    left_on="NEIGHBOR_CELL",
    right_on="CELL_ID",
    how="left"
)

In [ ]:
merged = merged.rename(columns={"STATUS": "NEIGHBOR_STATUS"})

merged = merged.drop(columns=["CELL_ID_y"])

merged = merged.rename(columns={"CELL_ID_x": "CELL_ID"})

In [ ]:
merged["STRONG_EVIDENCE"] = (
    (merged["CELL_STATUS"] == "Critical") &
    (merged["NEIGHBOR_STATUS"] == "Healthy")
)

In [ ]:
merged["STRONG_EVIDENCE"] = (
    (merged["CELL_STATUS"] == "Critical") &
    (merged["NEIGHBOR_STATUS"] == "Healthy")
)

In [ ]:
merged.head()

In [ ]:
final_output = merged[merged["STRONG_EVIDENCE"] == True]

In [ ]:
final_output.head()

In [ ]:
print("Total Strong Evidence Cases:", len(final_output))

In [ ]:
final_output.to_csv("strong_evidence_output.csv", index=False)

In [ ]:
final_output = final_output.drop_duplicates(
    subset=["CELL_ID", "NEIGHBOR_CELL"]
)

In [ ]:
final_output.head()

In [ ]:
final_output = final_output.sort_values(by="DISTANCE")

In [ ]:
final_output.to_excel("FINAL_STRONG_EVIDENCE.xlsx", index=False)

In [ ]:
# Unique problematic cells
final_strong = final_output[final_output["STRONG_EVIDENCE"] == True]

problem_cells = final_strong["CELL_ID"].unique()

print("Problematic Cells:", problem_cells)

In [ ]:
final_strong["CELL_ID"].value_counts()

In [ ]:
# Step 1: Top 10 most problematic cells
top_10 = final_strong["CELL_ID"].value_counts().head(10)
print("Top 10 Problematic Cells:\n", top_10)

# Step 2: Total problematic cells
problem_cells = final_strong["CELL_ID"].unique()
print("\nTotal Problematic Cells:", len(problem_cells))

# Step 3: Save final file
final_strong.to_csv("final_problematic_cells.csv", index=False)

In [ ]:
import pandas as pd
import numpy as np

file_path = r"C:\Users\malik\AppData\Local\Packages\5319275A.WhatsAppDesktop_cv1g1gvanyjgm\LocalState\sessions\711B537ADBC4AEA5082BDAB60FD84D562764D598\transfers\2026-25\LAC_CELL_CIRCLE_2G_SITES_DATA_FINAL.xlsx"

df1 = pd.read_excel(file_path, sheet_name=0)
df2 = pd.read_excel(file_path, sheet_name=1)
df3 = pd.read_excel(file_path, sheet_name=2)

df1["PERIOD"] = "May 8_10"
df2["PERIOD"] = "May 15_17"
df3["PERIOD"] = "May 22_24"

master_df = pd.concat([df1, df2, df3], ignore_index=True)

# column cleaning
master_df.columns = (
    master_df.columns
    .str.strip()
    .str.upper()
    .str.replace(" ", "_")
)

In [ ]:
pivot_df = master_df.pivot_table(
    index=['CIRCLE_NAME','LAC_ID','CELL_ID'],
    columns='PERIOD',
    values='SIM_COUNT',
    aggfunc='sum'
).reset_index()

pivot_df.columns.name = None

pivot_df['PCT_CHANGE'] = (
    (pivot_df['May 22_24'] - pivot_df['May 8_10']) 
    / pivot_df['May 8_10']
) * 100

pivot_df['FLAG'] = np.where(
    pivot_df['PCT_CHANGE'] < -30,
    'DROP',
    'OK'
)

final_strong = pivot_df[pivot_df['FLAG'] == 'DROP']

top_10 = final_strong.sort_values('PCT_CHANGE').head(10)

print(top_10)

In [ ]:
final_strong.groupby("CIRCLE_NAME")["CELL_ID"].count()

In [ ]:
import pandas as pd

master_df = pd.read_csv("master_dataset.csv")

master_df.columns = (
    master_df.columns
    .str.strip()
    .str.upper()
    .str.replace(" ", "_")
)

pivot_df = master_df.pivot_table(
    index=["CIRCLE_NAME", "LAC_ID", "CELL_ID"],
    columns="PERIOD",
    values="SIM_COUNT",
    aggfunc="sum"
).reset_index()

pivot_df.columns.name = None

baseline = "May 8_10"
mid = "May 15_17"
current = "May 22_24"

pivot_df["ROLLING_AVG"] = pivot_df[[baseline, mid]].mean(axis=1)

pivot_df["PCT_CHANGE_ROLLING"] = (
    (pivot_df[current] - pivot_df["ROLLING_AVG"])
    / pivot_df["ROLLING_AVG"]
) * 100

pivot_df["PCT_CHANGE_PREV"] = (
    (pivot_df[current] - pivot_df[mid])
    / pivot_df[mid]
) * 100

def flag_logic(row):
    # 🔴 STRONG DROP (50% se zyada)
    if row["PCT_CHANGE_ROLLING"] < -50 and row["PCT_CHANGE_PREV"] < -50:
        return "STRONG_DROP"
    
    # ⚠️ MODERATE DROP (30% se zyada)
    elif row["PCT_CHANGE_ROLLING"] < -30 or row["PCT_CHANGE_PREV"] < -30:
        return "MODERATE_DROP"
    
    else:
        return "NORMAL"

pivot_df["FLAG"] = pivot_df.apply(flag_logic, axis=1)

final_df = pivot_df[pivot_df["FLAG"] != "NORMAL"]

final_df.to_csv("final_problematic_cells.csv", index=False)

summary = final_df.groupby("CIRCLE_NAME")["CELL_ID"].nunique()

print("Problematic Cells by Circle:\n")
print(summary)

print("\nTotal Problematic Cells:", final_df["CELL_ID"].nunique())

In [ ]:
print(pivot_df.columns)

In [ ]:
print(pivot_df["FLAG"].value_counts())

In [ ]:
# sort by worst drop (rolling comparison)
top_worst = pivot_df.sort_values(by="PCT_CHANGE_ROLLING").head(10)

top_worst[['CIRCLE_NAME','LAC_ID','CELL_ID',
           'ROLLING_AVG','May 22_24',
           'PCT_CHANGE_ROLLING','FLAG']]

In [ ]:
strong_df = pivot_df[pivot_df["FLAG"] == "STRONG_DROP"]

circle_summary = strong_df.groupby("CIRCLE_NAME")["CELL_ID"].nunique()

print(circle_summary)

In [ ]:
circle_total = pivot_df.groupby("CIRCLE_NAME")["CELL_ID"].nunique()

impact_df = (circle_summary / circle_total * 100).reset_index()
impact_df.columns = ["CIRCLE_NAME", "IMPACT_%"]

impact_df.sort_values(by="IMPACT_%", ascending=False)

In [ ]:
pivot_df[["PCT_CHANGE_ROLLING","PCT_CHANGE_PREV"]].describe()

In [ ]:
import matplotlib.pyplot as plt

pivot_df["PCT_CHANGE_ROLLING"].hist(bins=50)
plt.title("Distribution of % Drop (Rolling Avg)")
plt.xlabel("% Change")
plt.ylabel("Frequency")
plt.show()

In [ ]:
top_worst[['CELL_ID','PCT_CHANGE_ROLLING']].head(5)

In [ ]:
top_worst["LOSS"] = top_worst["ROLLING_AVG"] - top_worst["May 22_24"]

top_worst[["CELL_ID","LOSS"]]

In [ ]:
# 1. create column
pivot_df["LOSS"] = pivot_df["ROLLING_AVG"] - pivot_df["May 22_24"]

# 2. filter
strong_df = pivot_df[pivot_df["FLAG"] == "STRONG_DROP"]

# 3. calculate
total_loss = strong_df["LOSS"].sum()

print("Total Users Impacted:", total_loss)

In [ ]:
list(globals().keys())

In [ ]:
import pandas as pd

# ================================
# 1. LOAD DATA
# ================================
pivot_df = master_df.copy()

# ================================
# 2. LONG → PIVOT (IMPORTANT FIX)
# ================================
pivot_df = pivot_df.pivot_table(
    index=['CIRCLE_NAME', 'LAC_ID', 'CELL_ID'],
    columns='PERIOD',
    values='SESSION_COUNT',
    aggfunc='sum'
).reset_index()

# Clean column names
pivot_df.columns = pivot_df.columns.astype(str)
pivot_df.columns = pivot_df.columns.str.strip()

# ================================
# 3. CHECK REQUIRED COLUMNS
# ================================
required_cols = ['May 8_10', 'May 15_17', 'May 22_24']

for col in required_cols:
    if col not in pivot_df.columns:
        raise Exception(f"Column missing: {col}")

# ================================
# 4. CALCULATIONS
# ================================
pivot_df['ROLLING_AVG'] = pivot_df[['May 8_10', 'May 15_17']].mean(axis=1)

pivot_df['PCT_CHANGE_ROLLING'] = (
    (pivot_df['May 22_24'] - pivot_df['ROLLING_AVG']) 
    / pivot_df['ROLLING_AVG']
) * 100

pivot_df['PCT_CHANGE_PREV'] = (
    (pivot_df['May 22_24'] - pivot_df['May 15_17']) 
    / pivot_df['May 15_17']
) * 100

# ================================
# 5. FLAG LOGIC (50% DROP)
# ================================
def flag_logic(x):
    if x <= -50:
        return 'STRONG_DROP'
    elif x <= -20:
        return 'MODERATE_DROP'
    else:
        return 'NORMAL'

pivot_df['FLAG'] = pivot_df['PCT_CHANGE_ROLLING'].apply(flag_logic)

# ================================
# 6. LOSS CALCULATION
# ================================
pivot_df['LOSS'] = pivot_df['ROLLING_AVG'] - pivot_df['May 22_24']

# ================================
# 7. STRONG DROP TABLE
# ================================
strong_df = pivot_df[pivot_df['FLAG'] == 'STRONG_DROP'].copy()

# ================================
# 8. FINAL IMPACT TABLE (MERGED)
# ================================
circle_impact = (
    strong_df.groupby('CIRCLE_NAME')['LOSS']
    .sum()
    .reset_index()
    .sort_values(by='LOSS', ascending=False)
)

# ================================
# 9. FINAL OUTPUT
# ================================
print("Final Tables:")
print("1. pivot_df (Main Table)")
print("2. strong_df (Critical Cells)")
print("3. circle_impact (Final Summary)")

print("\nTotal Strong Drop Cells:", strong_df.shape[0])
print("Total Loss:", strong_df['LOSS'].sum())

In [ ]:
import pandas as pd

df = pd.read_excel(r"C:\Users\malik\AppData\Local\Packages\5319275A.WhatsAppDesktop_cv1g1gvanyjgm\LocalState\sessions\711B537ADBC4AEA5082BDAB60FD84D562764D598\transfers\2026-25\LAC_CELL_CIRCLE_2G_SITES_DATA_FINAL.xlsx")

df.head()

In [ ]:
import pandas as pd

df = pd.read_excel(r"C:\Users\malik\Downloads\LAC_CELL_CIRCLE_2G_SITES_DATA_FINAL.xlsx")  

# STEP 2: CORRECT LOSS (important fix)
df['loss'] = df['sim_count'] - df['session_count']

# STEP 3: PIVOT TABLE
pivot_df = df.pivot_table(
    values='loss',
    index='LAC ID',
    columns='CELL ID',
    aggfunc='sum'
)

# STEP 4: STRONG DROP CONDITION
mask = pivot_df < -1000

# STEP 5: FINAL RESULT
circle_impact = {
    "Total Strong Drop Cells": int(mask.sum().sum()),
    "Total Loss": float(pivot_df.where(mask).sum().sum())
}

print(circle_impact)

In [ ]:
print(df['loss'].describe())

In [ ]:
top_cells = df.sort_values('loss').head(10)
print(top_cells)

In [ ]:
print(top_cells)

In [ ]:
circle_summary = df.groupby('circle_name')['loss'].sum()
print(circle_summary)

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

plt.figure(figsize=(12,6))
sns.heatmap(pivot_df, cmap='coolwarm')
plt.show()

In [ ]:
import pandas as pd

# Load data
df = pd.read_excel(r"C:\Users\malik\AppData\Local\Packages\5319275A.WhatsAppDesktop_cv1g1gvanyjgm\LocalState\sessions\711B537ADBC4AEA5082BDAB60FD84D562764D598\transfers\2026-25\LAC_CELL_CIRCLE_2G_SITES_DATA_FINAL.xlsx")


# Create loss column
df['loss'] = df['sim_count'] - df['session_count']

# LAC level aggregation
lac_summary = df.groupby('LAC ID')['loss'].sum().reset_index()
lac_summary.rename(columns={'loss': 'lac_loss'}, inplace=True)

# Circle level aggregation
circle_summary = df.groupby('circle_name')['loss'].sum().reset_index()
circle_summary.rename(columns={'loss': 'circle_loss'}, inplace=True)

# Merge all into one final table
final_df = df.merge(lac_summary, on='LAC ID', how='left')
final_df = final_df.merge(circle_summary, on='circle_name', how='left')

# Final output
print(final_df.head())
print(final_df.shape)

In [ ]:
# avoid division error
df['sim_pct_change'] = df['loss'] / df['sim_count'].replace(0, 1) * 100
df['session_pct_change'] = df['loss'] / df['session_count'].replace(0, 1) * 100

# suspicion score (better than flag)
df['suspicion_score'] = (
    abs(df['sim_pct_change']) * 0.5 +
    abs(df['session_pct_change']) * 0.5
)

# classification
df['suspicion_level'] = pd.cut(
    df['suspicion_score'],
    bins=[0, 20, 50, 100, 1000],
    labels=['Low', 'Medium', 'High', 'Critical']
)

In [ ]:
df.columns

In [ ]:
df.head()

In [ ]:
df.isnull().sum()

In [ ]:
df['suspicion_level'] = pd.cut(
    df['suspicion_score'],
    bins=[-float('inf'), 20, 50, 100, float('inf')],
    labels=['Low', 'Medium', 'High', 'Critical']
)

In [ ]:
df['suspicion_level'].isnull().sum()

In [ ]:
import pandas as pd

# load master data
master_df = pd.read_csv("master_dataset.csv")

# clean columns
master_df.columns = (
    master_df.columns
    .str.strip()
    .str.upper()
    .str.replace(" ", "_")
)

# pivot (SIM_COUNT use kar rahe hain)
pivot_df = master_df.pivot_table(
    index=["CIRCLE_NAME", "LAC_ID", "CELL_ID"],
    columns="PERIOD",
    values="SIM_COUNT",
    aggfunc="sum"
).reset_index()

pivot_df.columns.name = None

# define periods
baseline = "May 8_10"
mid = "May 15_17"
current = "May 22_24"

# rolling avg
pivot_df["ROLLING_AVG"] = pivot_df[[baseline, mid]].mean(axis=1)

# % change
pivot_df["PCT_CHANGE_ROLLING"] = (
    (pivot_df[current] - pivot_df["ROLLING_AVG"])
    / pivot_df["ROLLING_AVG"]
) * 100

# flag logic
def flag_logic(row):
    if row["PCT_CHANGE_ROLLING"] < -50:
        return "STRONG_DROP"
    elif row["PCT_CHANGE_ROLLING"] < -30:
        return "MODERATE_DROP"
    else:
        return "NORMAL"

pivot_df["FLAG"] = pivot_df.apply(flag_logic, axis=1)

# loss
pivot_df["LOSS"] = pivot_df["ROLLING_AVG"] - pivot_df[current]

# FINAL TABLE
final_df = pivot_df.copy()

# save
final_df.to_csv("FINAL_DASHBOARD_TABLE.csv", index=False)

print("✅ Final table ready!")
print(final_df.head())

In [ ]:
# fill missing values
pivot_df = pivot_df.fillna(0)

# rolling avg
pivot_df["ROLLING_AVG"] = pivot_df[[baseline, mid]].mean(axis=1)
pivot_df["ROLLING_AVG"] = pivot_df["ROLLING_AVG"].replace(0, 1)

# % change
pivot_df["PCT_CHANGE_ROLLING"] = (
    (pivot_df[current] - pivot_df["ROLLING_AVG"])
    / pivot_df["ROLLING_AVG"]
) * 100

# loss
pivot_df["LOSS"] = pivot_df["ROLLING_AVG"] - pivot_df[current]

In [ ]:
pivot_df.isnull().sum()

In [ ]:
print(final_df.head())

In [ ]:
pivot_df = pivot_df.dropna()

In [ ]:
print(final_df.head())

In [ ]:
# STEP 1: Fill missing values
pivot_df = pivot_df.fillna(0)

# STEP 2: Define columns
baseline = "May 8_10"
mid = "May 15_17"
current = "May 22_24"

# STEP 3: Rolling Average
pivot_df["ROLLING_AVG"] = pivot_df[[baseline, mid]].mean(axis=1)

# avoid divide by zero
pivot_df["ROLLING_AVG"] = pivot_df["ROLLING_AVG"].replace(0, 1)

# STEP 4: % Change
pivot_df["PCT_CHANGE_ROLLING"] = (
    (pivot_df[current] - pivot_df["ROLLING_AVG"])
    / pivot_df["ROLLING_AVG"]
) * 100

# STEP 5: LOSS
pivot_df["LOSS"] = pivot_df["ROLLING_AVG"] - pivot_df[current]

# STEP 6: FLAG
def flag(x):
    if x <= -70:
        return "STRONG_DROP"
    elif x <= -40:
        return "MODERATE_DROP"
    else:
        return "NORMAL"

pivot_df["FLAG"] = pivot_df["PCT_CHANGE_ROLLING"].apply(flag)

# FINAL TABLE
final_table = pivot_df

# SHOW OUTPUT
print(final_table.head())
print("\nNull Check:\n", final_table.isnull().sum())

In [ ]:
import pandas as pd
import numpy as np
from sklearn.neighbors import NearestNeighbors

# ================================
# 1. LOAD DATA
# ================================
file_path = r"C:\Users\malik\Downloads\LAC_CELL_CIRCLE_2G_SITES_DATA_FINAL.xlsx"

df1 = pd.read_excel(file_path, sheet_name=0)
df2 = pd.read_excel(file_path, sheet_name=1)
df3 = pd.read_excel(file_path, sheet_name=2)

df1["PERIOD"] = "May 8_10"
df2["PERIOD"] = "May 15_17"
df3["PERIOD"] = "May 22_24"

master_df = pd.concat([df1, df2, df3], ignore_index=True)

# clean columns
master_df.columns = (
    master_df.columns
    .str.strip()
    .str.upper()
    .str.replace(" ", "_")
)

# ================================
# 2. PIVOT (SESSION BASED)
# ================================
pivot_df = master_df.pivot_table(
    index=['CIRCLE_NAME', 'LAC_ID', 'CELL_ID'],
    columns='PERIOD',
    values='SESSION_COUNT',
    aggfunc='sum'
).reset_index()

pivot_df.columns.name = None

# ================================
# 3. HANDLE MISSING VALUES
# ================================
pivot_df = pivot_df.fillna(0)

baseline = "May 8_10"
mid = "May 15_17"
current = "May 22_24"

# ================================
# 4. CALCULATIONS
# ================================
pivot_df["ROLLING_AVG"] = pivot_df[[baseline, mid]].mean(axis=1)

pivot_df["ROLLING_AVG"] = pivot_df["ROLLING_AVG"].replace(0, 1)

pivot_df["PCT_CHANGE_ROLLING"] = (
    (pivot_df[current] - pivot_df["ROLLING_AVG"])
    / pivot_df["ROLLING_AVG"]
) * 100

pivot_df["LOSS"] = pivot_df["ROLLING_AVG"] - pivot_df[current]

# ================================
# 5. FLAG LOGIC
# ================================
def flag(x):
    if x <= -70:
        return "STRONG_DROP"
    elif x <= -40:
        return "MODERATE_DROP"
    else:
        return "NORMAL"

pivot_df["FLAG"] = pivot_df["PCT_CHANGE_ROLLING"].apply(flag)

# ================================
# 6. LAT-LONG GENERATION
# ================================
np.random.seed(42)

coords_df = pivot_df[['CELL_ID', 'LAC_ID']].copy()

coords_df["LAT"] = np.random.uniform(28.40, 28.90, len(coords_df))
coords_df["LONG"] = np.random.uniform(76.90, 77.40, len(coords_df))

# merge coords
pivot_df = pivot_df.merge(coords_df, on=["CELL_ID", "LAC_ID"], how="left")

# ================================
# 7. NEIGHBOR ANALYSIS
# ================================
coords = pivot_df[["LAT", "LONG"]]

nn = NearestNeighbors(n_neighbors=6, algorithm='ball_tree')
nn.fit(coords)

distances, indices = nn.kneighbors(coords)

neighbor_data = []

for i in range(len(pivot_df)):
    for j in range(1, 6):
        neighbor_data.append({
            "CELL_ID": pivot_df.iloc[i]["CELL_ID"],
            "NEIGHBOR_CELL": pivot_df.iloc[indices[i][j]]["CELL_ID"],
            "DISTANCE": distances[i][j]
        })

neighbor_df = pd.DataFrame(neighbor_data)

# ================================
# 8. STATUS MAPPING
# ================================
status_df = pivot_df[["CELL_ID", "FLAG"]].copy()
status_df = status_df.rename(columns={"FLAG": "CELL_STATUS"})

# merge cell status
merged = neighbor_df.merge(status_df, on="CELL_ID", how="left")

# merge neighbor status
merged = merged.merge(
    status_df,
    left_on="NEIGHBOR_CELL",
    right_on="CELL_ID",
    how="left"
)

merged = merged.rename(columns={"CELL_STATUS_x": "CELL_STATUS",
                                "CELL_STATUS_y": "NEIGHBOR_STATUS"})

merged = merged.drop(columns=["CELL_ID_y"])
merged = merged.rename(columns={"CELL_ID_x": "CELL_ID"})

# ================================
# 9. STRONG EVIDENCE LOGIC
# ================================
merged["STRONG_EVIDENCE"] = (
    (merged["CELL_STATUS"] == "STRONG_DROP") &
    (merged["NEIGHBOR_STATUS"] == "NORMAL")
)

# ================================
# 10. FINAL TABLE (ALL IN ONE)
# ================================
final_table = pivot_df.merge(
    merged[["CELL_ID", "NEIGHBOR_CELL", "DISTANCE", "STRONG_EVIDENCE"]],
    on="CELL_ID",
    how="left"
)

# ================================
# 11. SORTING
# ================================
final_table = final_table.sort_values(by="PCT_CHANGE_ROLLING")

# ================================
# 12. EXPORT FINAL DATA
# ================================
final_table.to_csv("FINAL_DASHBOARD_TABLE.csv", index=False)

print(final_table.head())

In [ ]:
# ================================
# 13. REMOVE DUPLICATES (IMPORTANT FIX)
# ================================
final_table = final_table.sort_values(by="PCT_CHANGE_ROLLING")

final_table = final_table.drop_duplicates(
    subset=["CELL_ID", "NEIGHBOR_CELL"],
    keep="first"
)

# ================================
# 14. FINAL STATUS (ROOT CAUSE)
# ================================
def final_status(row):
    if row["FLAG"] == "STRONG_DROP" and row["STRONG_EVIDENCE"] == True:
        return "TRAFFIC_SHIFTED"
    elif row["FLAG"] == "STRONG_DROP":
        return "POSSIBLE_OUTAGE"
    elif row["FLAG"] == "MODERATE_DROP":
        return "MONITOR"
    else:
        return "NORMAL"

final_table["FINAL_STATUS"] = final_table.apply(final_status, axis=1)

# ================================
# 15. PRIORITY LEVEL
# ================================
def priority(loss):
    if loss >= 5:
        return "HIGH"
    elif loss >= 1:
        return "MEDIUM"
    else:
        return "LOW"

final_table["PRIORITY"] = final_table["LOSS"].apply(priority)

# ================================
# 16. IMPACT SCORE (SMART KPI 🔥)
# ================================
final_table["IMPACT_SCORE"] = (
    abs(final_table["PCT_CHANGE_ROLLING"]) * final_table["LOSS"]
)

# ================================
# 17. CLEAN FINAL COLUMNS
# ================================
final_columns = [
    "CIRCLE_NAME",
    "LAC_ID",
    "CELL_ID",
    "May 8_10",
    "May 15_17",
    "May 22_24",
    "ROLLING_AVG",
    "PCT_CHANGE_ROLLING",
    "LOSS",
    "FLAG",
    "FINAL_STATUS",
    "PRIORITY",
    "IMPACT_SCORE",
    "LAT",
    "LONG",
    "NEIGHBOR_CELL",
    "DISTANCE",
    "STRONG_EVIDENCE"
]

final_table = final_table[final_columns]

# ================================
# 18. SORT FOR DASHBOARD
# ================================
final_table = final_table.sort_values(
    by=["PRIORITY", "IMPACT_SCORE"],
    ascending=[True, False]
)

# ================================
# 19. SAVE FINAL OUTPUT
# ================================
final_table.to_csv("FINAL_SMART_DASHBOARD.csv", index=False)

# ================================
# 20. QUICK SUMMARY PRINT
# ================================
print("✅ FINAL DATA READY")

print("\n📊 STATUS COUNT:")
print(final_table["FINAL_STATUS"].value_counts())

print("\n🔥 TOP ISSUES:")
print(final_table.head(10))

In [ ]:
# ================================
# 21. BEST NEIGHBOR PER CELL (FIX DUPLICATES)
# ================================

# priority: strong evidence first, then nearest distance
final_table_sorted = final_table.sort_values(
    by=["CELL_ID", "STRONG_EVIDENCE", "DISTANCE"],
    ascending=[True, False, True]
)

# keep best row per cell
final_unique = final_table_sorted.drop_duplicates(
    subset=["CELL_ID"],
    keep="first"
)

# ================================
# 22. RE-CALCULATE STATUS CLEANLY
# ================================
def clean_status(row):
    if row["FLAG"] == "STRONG_DROP" and row["STRONG_EVIDENCE"]:
        return "TRAFFIC_SHIFTED"
    elif row["FLAG"] == "STRONG_DROP":
        return "POSSIBLE_OUTAGE"
    elif row["FLAG"] == "MODERATE_DROP":
        return "MONITOR"
    else:
        return "NORMAL"

final_unique["FINAL_STATUS"] = final_unique.apply(clean_status, axis=1)

# ================================
# 23. SORT FINAL OUTPUT
# ================================
final_unique = final_unique.sort_values(
    by=["PRIORITY", "IMPACT_SCORE"],
    ascending=[True, False]
)

# ================================
# 24. SAVE CLEAN FINAL FILE
# ================================
final_unique.to_csv("FINAL_CLEAN_DASHBOARD.csv", index=False)

# ================================
# 25. FINAL CHECK
# ================================
print("✅ CLEAN FINAL READY")

print("\n📊 UNIQUE CELL COUNT:")
print(final_unique["CELL_ID"].nunique())

print("\n📊 STATUS COUNT:")
print(final_unique["FINAL_STATUS"].value_counts())

print("\n🔥 TOP 10 ISSUES:")
print(final_unique.head(10))

In [ ]:
import pandas as pd
import numpy as np
from sklearn.neighbors import NearestNeighbors

# ================================
# 1. LOAD DATA
# ================================
file_path = r"C:\Users\malik\Downloads\LAC_CELL_CIRCLE_2G_SITES_DATA_FINAL.xlsx"

df1 = pd.read_excel(file_path, sheet_name=0)
df2 = pd.read_excel(file_path, sheet_name=1)
df3 = pd.read_excel(file_path, sheet_name=2)

df1["PERIOD"] = "May 8_10"
df2["PERIOD"] = "May 15_17"
df3["PERIOD"] = "May 22_24"

master_df = pd.concat([df1, df2, df3], ignore_index=True)

# ================================
# 2. CLEAN COLUMNS
# ================================
master_df.columns = (
    master_df.columns
    .str.strip()
    .str.upper()
    .str.replace(" ", "_")
)

# ================================
# 3. PIVOT TABLE
# ================================
pivot_df = master_df.pivot_table(
    index=['CIRCLE_NAME', 'LAC_ID', 'CELL_ID'],
    columns='PERIOD',
    values='SESSION_COUNT',
    aggfunc='sum'
).reset_index()

pivot_df = pivot_df.fillna(0)

baseline = "May 8_10"
mid = "May 15_17"
current = "May 22_24"

# ================================
# 4. CALCULATIONS
# ================================
pivot_df["ROLLING_AVG"] = pivot_df[[baseline, mid]].mean(axis=1)
pivot_df["ROLLING_AVG"] = pivot_df["ROLLING_AVG"].replace(0, 1)

pivot_df["PCT_CHANGE_ROLLING"] = (
    (pivot_df[current] - pivot_df["ROLLING_AVG"])
    / pivot_df["ROLLING_AVG"]
) * 100

pivot_df["LOSS"] = pivot_df["ROLLING_AVG"] - pivot_df[current]

# ================================
# 5. FLAG
# ================================
def flag(x):
    if x <= -70:
        return "STRONG_DROP"
    elif x <= -40:
        return "MODERATE_DROP"
    else:
        return "NORMAL"

pivot_df["FLAG"] = pivot_df["PCT_CHANGE_ROLLING"].apply(flag)

# ================================
# 6. LAT-LONG (SIMULATED)
# ================================
np.random.seed(42)

coords_df = pivot_df[['CELL_ID', 'LAC_ID']].copy()
coords_df["LAT"] = np.random.uniform(28.40, 28.90, len(coords_df))
coords_df["LONG"] = np.random.uniform(76.90, 77.40, len(coords_df))

pivot_df = pivot_df.merge(coords_df, on=["CELL_ID", "LAC_ID"], how="left")

# ================================
# 7. NEIGHBOR ANALYSIS
# ================================
coords = pivot_df[["LAT", "LONG"]]

nn = NearestNeighbors(n_neighbors=6)
nn.fit(coords)

distances, indices = nn.kneighbors(coords)

neighbor_data = []

for i in range(len(pivot_df)):
    for j in range(1, 6):
        neighbor_data.append({
            "CELL_ID": pivot_df.iloc[i]["CELL_ID"],
            "NEIGHBOR_CELL": pivot_df.iloc[indices[i][j]]["CELL_ID"],
            "DISTANCE": distances[i][j]
        })

neighbor_df = pd.DataFrame(neighbor_data)

# ================================
# 8. STATUS MERGE
# ================================
status_df = pivot_df[["CELL_ID", "FLAG"]].copy()
status_df = status_df.rename(columns={"FLAG": "CELL_STATUS"})

merged = neighbor_df.merge(status_df, on="CELL_ID", how="left")

merged = merged.merge(
    status_df,
    left_on="NEIGHBOR_CELL",
    right_on="CELL_ID",
    how="left"
)

merged = merged.rename(columns={
    "CELL_STATUS_x": "CELL_STATUS",
    "CELL_STATUS_y": "NEIGHBOR_STATUS"
})

merged = merged.drop(columns=["CELL_ID_y"])
merged = merged.rename(columns={"CELL_ID_x": "CELL_ID"})

# ================================
# 9. STRONG EVIDENCE
# ================================
merged["STRONG_EVIDENCE"] = (
    (merged["CELL_STATUS"] == "STRONG_DROP") &
    (merged["NEIGHBOR_STATUS"] == "NORMAL")
)

# ================================
# 10. MERGE BACK
# ================================
final_table = pivot_df.merge(
    merged[["CELL_ID", "NEIGHBOR_CELL", "DISTANCE", "STRONG_EVIDENCE"]],
    on="CELL_ID",
    how="left"
)

# ================================
# 11. PRIORITY
# ================================
def priority(loss):
    if loss >= 5000:
        return "HIGH"
    elif loss >= 1000:
        return "MEDIUM"
    else:
        return "LOW"

final_table["PRIORITY"] = final_table["LOSS"].apply(priority)

# ================================
# 12. IMPACT SCORE
# ================================
final_table["IMPACT_SCORE"] = (
    abs(final_table["PCT_CHANGE_ROLLING"]) * final_table["LOSS"]
)

# ================================
# 13. BEST NEIGHBOR (REMOVE DUPLICATES)
# ================================
final_table = final_table.sort_values(
    by=["CELL_ID", "STRONG_EVIDENCE", "DISTANCE"],
    ascending=[True, False, True]
)

final_unique = final_table.drop_duplicates(subset=["CELL_ID"]).copy()

# ================================
# 14. FINAL STATUS
# ================================
def final_status(row):
    if row["FLAG"] == "STRONG_DROP" and row["STRONG_EVIDENCE"]:
        return "TRAFFIC_SHIFTED"
    elif row["FLAG"] == "STRONG_DROP":
        return "POSSIBLE_OUTAGE"
    elif row["FLAG"] == "MODERATE_DROP":
        return "MONITOR"
    else:
        return "NORMAL"

final_unique["FINAL_STATUS"] = final_unique.apply(final_status, axis=1)

# ================================
# 15. ACTION COLUMN
# ================================
final_unique["ACTION"] = final_unique["FINAL_STATUS"].map({
    "TRAFFIC_SHIFTED": "No Action Needed",
    "POSSIBLE_OUTAGE": "Immediate Check Required",
    "MONITOR": "Keep Monitoring",
    "NORMAL": "Healthy"
})

# ================================
# 16. FINAL SORT
# ================================
final_unique = final_unique.sort_values(
    by=["PRIORITY", "IMPACT_SCORE"],
    ascending=[True, False]
)

# ================================
# 17. EXPORT
# ================================
final_unique.to_csv("FINAL_TELECOM_DASHBOARD.csv", index=False)

# ================================
# 18. SUMMARY
# ================================

print("\n📊 TOTAL CELLS:", final_unique["CELL_ID"].nunique())

print("\n📊 STATUS COUNT:")
print(final_unique["FINAL_STATUS"].value_counts())

print("\n🔥 TOP 10 ISSUES:")
print(final_unique.head(10))

In [ ]:
import pandas as pd
import numpy as np
from sklearn.neighbors import NearestNeighbors

# ================================
# 1. LOAD DATA
# ================================
file_path = r"C:\Users\malik\Downloads\LAC_CELL_CIRCLE_2G_SITES_DATA_FINAL.xlsx"

df1 = pd.read_excel(file_path, sheet_name=0)
df2 = pd.read_excel(file_path, sheet_name=1)
df3 = pd.read_excel(file_path, sheet_name=2)

df1["PERIOD"] = "May 8_10"
df2["PERIOD"] = "May 15_17"
df3["PERIOD"] = "May 22_24"

master_df = pd.concat([df1, df2, df3], ignore_index=True)

# ================================
# 2. CLEAN COLUMNS
# ================================
master_df.columns = (
    master_df.columns
    .str.strip()
    .str.upper()
    .str.replace(" ", "_")
)

# ================================
# 3. PIVOT TABLE
# ================================
pivot_df = master_df.pivot_table(
    index=['CIRCLE_NAME', 'LAC_ID', 'CELL_ID'],
    columns='PERIOD',
    values='SESSION_COUNT',
    aggfunc='sum'
).reset_index()

pivot_df = pivot_df.fillna(0)

baseline = "May 8_10"
mid = "May 15_17"
current = "May 22_24"

# ================================
# 4. CALCULATIONS
# ================================
pivot_df["ROLLING_AVG"] = pivot_df[[baseline, mid]].mean(axis=1)
pivot_df["ROLLING_AVG"] = pivot_df["ROLLING_AVG"].replace(0, 1)

pivot_df["PCT_CHANGE_ROLLING"] = (
    (pivot_df[current] - pivot_df["ROLLING_AVG"])
    / pivot_df["ROLLING_AVG"]
) * 100

pivot_df["LOSS"] = pivot_df["ROLLING_AVG"] - pivot_df[current]

# ================================
# 5. FLAG
# ================================
def flag(x):
    if x <= -70:
        return "STRONG_DROP"
    elif x <= -40:
        return "MODERATE_DROP"
    else:
        return "NORMAL"

pivot_df["FLAG"] = pivot_df["PCT_CHANGE_ROLLING"].apply(flag)

# ================================
# 6. LAT-LONG (SIMULATED)
# ================================
np.random.seed(42)

coords_df = pivot_df[['CELL_ID', 'LAC_ID']].copy()
coords_df["LAT"] = np.random.uniform(28.40, 28.90, len(coords_df))
coords_df["LONG"] = np.random.uniform(76.90, 77.40, len(coords_df))

pivot_df = pivot_df.merge(coords_df, on=["CELL_ID", "LAC_ID"], how="left")

# ================================
# 7. NEIGHBOR ANALYSIS
# ================================
coords = pivot_df[["LAT", "LONG"]]

nn = NearestNeighbors(n_neighbors=6)
nn.fit(coords)

distances, indices = nn.kneighbors(coords)

neighbor_data = []

for i in range(len(pivot_df)):
    for j in range(1, 6):
        neighbor_data.append({
            "CELL_ID": pivot_df.iloc[i]["CELL_ID"],
            "NEIGHBOR_CELL": pivot_df.iloc[indices[i][j]]["CELL_ID"],
            "DISTANCE": distances[i][j]
        })

neighbor_df = pd.DataFrame(neighbor_data)

# ================================
# 8. STATUS MERGE
# ================================
status_df = pivot_df[["CELL_ID", "FLAG"]].copy()
status_df = status_df.rename(columns={"FLAG": "CELL_STATUS"})

merged = neighbor_df.merge(status_df, on="CELL_ID", how="left")

merged = merged.merge(
    status_df,
    left_on="NEIGHBOR_CELL",
    right_on="CELL_ID",
    how="left"
)

merged = merged.rename(columns={
    "CELL_STATUS_x": "CELL_STATUS",
    "CELL_STATUS_y": "NEIGHBOR_STATUS"
})

merged = merged.drop(columns=["CELL_ID_y"])
merged = merged.rename(columns={"CELL_ID_x": "CELL_ID"})

# ================================
# 9. STRONG EVIDENCE (IMPROVED)
# ================================
merged["STRONG_EVIDENCE"] = (
    (merged["CELL_STATUS"] == "STRONG_DROP") &
    (merged["NEIGHBOR_STATUS"] != "STRONG_DROP")
)

# ================================
# 10. MERGE BACK
# ================================
final_table = pivot_df.merge(
    merged[["CELL_ID", "NEIGHBOR_CELL", "DISTANCE", "STRONG_EVIDENCE"]],
    on="CELL_ID",
    how="left"
)

# ================================
# 11. PRIORITY (IMPROVED)
# ================================
def priority(loss):
    if loss >= 20000:
        return "CRITICAL"
    elif loss >= 5000:
        return "HIGH"
    elif loss >= 1000:
        return "MEDIUM"
    else:
        return "LOW"

final_table["PRIORITY"] = final_table["LOSS"].apply(priority)

# ================================
# 12. IMPACT SCORE
# ================================
final_table["IMPACT_SCORE"] = (
    abs(final_table["PCT_CHANGE_ROLLING"]) * final_table["LOSS"]
)

# ================================
# 13. BEST NEIGHBOR (REMOVE DUPLICATES)
# ================================
final_table = final_table.sort_values(
    by=["CELL_ID", "STRONG_EVIDENCE", "DISTANCE"],
    ascending=[True, False, True]
)

final_unique = final_table.drop_duplicates(subset=["CELL_ID"]).copy()

# ================================
# 14. FINAL STATUS
# ================================
def final_status(row):
    if row["FLAG"] == "STRONG_DROP" and row["STRONG_EVIDENCE"]:
        return "TRAFFIC_SHIFTED"
    elif row["FLAG"] == "STRONG_DROP":
        return "POSSIBLE_OUTAGE"
    elif row["FLAG"] == "MODERATE_DROP":
        return "MONITOR"
    else:
        return "NORMAL"

final_unique["FINAL_STATUS"] = final_unique.apply(final_status, axis=1)

# ================================
# 15. ACTION
# ================================
final_unique["ACTION"] = final_unique["FINAL_STATUS"].map({
    "TRAFFIC_SHIFTED": "No Action Needed",
    "POSSIBLE_OUTAGE": "Immediate Check Required",
    "MONITOR": "Keep Monitoring",
    "NORMAL": "Healthy"
})

# ================================
# 16. LOSS SHARE %
# ================================
final_unique["LOSS_SHARE_%"] = (
    final_unique["LOSS"] / final_unique["LOSS"].sum()
) * 100

# ================================
# 17. SORT (CORRECT ORDER)
# ================================
priority_order = {
    "CRITICAL": 1,
    "HIGH": 2,
    "MEDIUM": 3,
    "LOW": 4
}

final_unique["PRIORITY_RANK"] = final_unique["PRIORITY"].map(priority_order)

final_unique = final_unique.sort_values(
    by=["PRIORITY_RANK", "IMPACT_SCORE"],
    ascending=[True, False]
)

# ================================
# 18. EXPORT
# ================================
final_unique.to_csv("FINAL_TELECOM_DASHBOARD.csv", index=False)

# ================================
# 19. SUMMARY
# ================================

print("\n📊 TOTAL CELLS:", final_unique["CELL_ID"].nunique())

print("\n📊 STATUS COUNT:")
print(final_unique["FINAL_STATUS"].value_counts())

print("\n TOP 10 ISSUES:")
print(final_unique.head(10))

In [ ]:
import pandas as pd
import numpy as np
from sklearn.neighbors import NearestNeighbors

# ================================
# 1. LOAD DATA
# ================================
file_path = r"C:\Users\malik\Downloads\LAC_CELL_CIRCLE_2G_SITES_DATA_FINAL.xlsx"

df1 = pd.read_excel(file_path, sheet_name=0)
df2 = pd.read_excel(file_path, sheet_name=1)
df3 = pd.read_excel(file_path, sheet_name=2)

df1["PERIOD"] = "May 8_10"
df2["PERIOD"] = "May 15_17"
df3["PERIOD"] = "May 22_24"

master_df = pd.concat([df1, df2, df3], ignore_index=True)

# ================================
# 2. CLEAN COLUMNS
# ================================
master_df.columns = (
    master_df.columns
    .str.strip()
    .str.upper()
    .str.replace(" ", "_")
)

# ================================
# 3. PIVOT TABLE
# ================================
pivot_df = master_df.pivot_table(
    index=['CIRCLE_NAME', 'LAC_ID', 'CELL_ID'],
    columns='PERIOD',
    values='SESSION_COUNT',
    aggfunc='sum'
).reset_index()

pivot_df = pivot_df.fillna(0)

baseline = "May 8_10"
mid = "May 15_17"
current = "May 22_24"

# ================================
# 4. CALCULATIONS
# ================================
pivot_df["ROLLING_AVG"] = pivot_df[[baseline, mid]].mean(axis=1)
pivot_df["ROLLING_AVG"] = pivot_df["ROLLING_AVG"].replace(0, 1)

pivot_df["PCT_CHANGE_ROLLING"] = (
    (pivot_df[current] - pivot_df["ROLLING_AVG"])
    / pivot_df["ROLLING_AVG"]
) * 100

pivot_df["LOSS"] = pivot_df["ROLLING_AVG"] - pivot_df[current]

# ================================
# 5. FLAG
# ================================
def flag(x):
    if x <= -70:
        return "STRONG_DROP"
    elif x <= -40:
        return "MODERATE_DROP"
    else:
        return "NORMAL"

pivot_df["FLAG"] = pivot_df["PCT_CHANGE_ROLLING"].apply(flag)

# ================================
# 6. LAT-LONG (SIMULATED)
# ================================
np.random.seed(42)

coords_df = pivot_df[['CELL_ID', 'LAC_ID']].copy()
coords_df["LAT"] = np.random.uniform(28.40, 28.90, len(coords_df))
coords_df["LONG"] = np.random.uniform(76.90, 77.40, len(coords_df))

pivot_df = pivot_df.merge(coords_df, on=["CELL_ID", "LAC_ID"], how="left")

# ================================
# 7. NEIGHBOR ANALYSIS
# ================================
coords = pivot_df[["LAT", "LONG"]]

nn = NearestNeighbors(n_neighbors=6)
nn.fit(coords)

distances, indices = nn.kneighbors(coords)

neighbor_data = []

for i in range(len(pivot_df)):
    for j in range(1, 6):
        neighbor_data.append({
            "CELL_ID": pivot_df.iloc[i]["CELL_ID"],
            "NEIGHBOR_CELL": pivot_df.iloc[indices[i][j]]["CELL_ID"],
            "DISTANCE": distances[i][j]
        })

neighbor_df = pd.DataFrame(neighbor_data)

# ================================
# 8. STATUS MERGE
# ================================
status_df = pivot_df[["CELL_ID", "FLAG"]].copy()
status_df = status_df.rename(columns={"FLAG": "CELL_STATUS"})

merged = neighbor_df.merge(status_df, on="CELL_ID", how="left")

merged = merged.merge(
    status_df,
    left_on="NEIGHBOR_CELL",
    right_on="CELL_ID",
    how="left"
)

merged = merged.rename(columns={
    "CELL_STATUS_x": "CELL_STATUS",
    "CELL_STATUS_y": "NEIGHBOR_STATUS"
})

merged = merged.drop(columns=["CELL_ID_y"])
merged = merged.rename(columns={"CELL_ID_x": "CELL_ID"})

# ================================
# 9. STRONG EVIDENCE (IMPROVED)
# ================================
merged["STRONG_EVIDENCE"] = (
    (merged["CELL_STATUS"] == "STRONG_DROP") &
    (merged["NEIGHBOR_STATUS"] != "STRONG_DROP")
)

# ================================
# 10. MERGE BACK
# ================================
final_table = pivot_df.merge(
    merged[["CELL_ID", "NEIGHBOR_CELL", "DISTANCE", "STRONG_EVIDENCE"]],
    on="CELL_ID",
    how="left"
)

# ================================
# 11. PRIORITY (IMPROVED)
# ================================
def priority(loss):
    if loss >= 20000:
        return "CRITICAL"
    elif loss >= 5000:
        return "HIGH"
    elif loss >= 1000:
        return "MEDIUM"
    else:
        return "LOW"

final_table["PRIORITY"] = final_table["LOSS"].apply(priority)

# ================================
# 12. IMPACT SCORE
# ================================
final_table["IMPACT_SCORE"] = (
    abs(final_table["PCT_CHANGE_ROLLING"]) * final_table["LOSS"]
)

# ================================
# 13. BEST NEIGHBOR (REMOVE DUPLICATES)
# ================================
final_table = final_table.sort_values(
    by=["CELL_ID", "STRONG_EVIDENCE", "DISTANCE"],
    ascending=[True, False, True]
)

final_unique = final_table.drop_duplicates(subset=["CELL_ID"]).copy()

# ================================
# 14. FINAL STATUS
# ================================

def final_status(row):
    drop = row["PCT_CHANGE_ROLLING"]
    strong = row["STRONG_EVIDENCE"]
    impact = row["IMPACT_SCORE"]
    current_traffic = row["May 22_24"]

    # 1. SUSPECTED INACTIVE (highest confidence)
    if (drop <= -80) and strong and (current_traffic < 1000):
        return "Suspected Inactive"

    # 2. POSSIBLE LOCAL SHIFT
    elif (drop <= -70) and strong:
        return "Possible Local Shift"

    # 3. AREA-WIDE DECLINE
    elif (drop <= -70) and not strong:
        return "Area-wide Decline"

    # 4. MONITOR
    elif (drop <= -40):
        return "Monitor"

    # 5. HEALTHY
    else:
        return "Healthy"

final_unique["FINAL_STATUS"] = final_unique.apply(final_status, axis=1)

# ================================
# 15. ACTION
# ================================

def assign_action(status):
    if status == "Suspected Inactive":
        return "URGENT: Verify site, check outage/config"
    
    elif status == "Possible Local Shift":
        return "Check neighbor load & traffic migration"
    
    elif status == "Area-wide Decline":
        return "Review regional performance trends"
    
    elif status == "Monitor":
        return "Track next period"
    
    else:
        return "No action"

final_unique["ACTION"] = final_unique["FINAL_STATUS"].apply(assign_action)

# ================================
# 16. LOSS SHARE %
# ================================
final_unique["LOSS_SHARE_%"] = (
    final_unique["LOSS"] / final_unique["LOSS"].sum()
) * 100

# ================================
# 17. SORT (CORRECT ORDER)
# ================================
priority_order = {
    "CRITICAL": 1,
    "HIGH": 2,
    "MEDIUM": 3,
    "LOW": 4
}

final_unique["PRIORITY_RANK"] = final_unique["PRIORITY"].map(priority_order)

final_unique = final_unique.sort_values(
    by=["PRIORITY_RANK", "IMPACT_SCORE"],
    ascending=[True, False]
)

# ================================
# 18. EXPORT
# ================================
final_unique.to_csv("FINAL_TELECOM_DASHBOARD.csv", index=False)

# ================================
# 19. SUMMARY
# ================================

print("\n📊 TOTAL CELLS:", final_unique["CELL_ID"].nunique())

print("\n📊 STATUS COUNT:")
print(final_unique["FINAL_STATUS"].value_counts())

print("\n TOP 10 ISSUES:")
print(final_unique.head(10))

In [ ]:
import pandas as pd
import numpy as np
from sklearn.neighbors import NearestNeighbors

# ================================
# 1. LOAD DATA
# ================================
file_path = r"C:\Users\malik\Downloads\LAC_CELL_CIRCLE_2G_SITES_DATA_FINAL.xlsx"

df1 = pd.read_excel(file_path, sheet_name=0)
df2 = pd.read_excel(file_path, sheet_name=1)
df3 = pd.read_excel(file_path, sheet_name=2)

df1["PERIOD"] = "May 8_10"
df2["PERIOD"] = "May 15_17"
df3["PERIOD"] = "May 22_24"

master_df = pd.concat([df1, df2, df3], ignore_index=True)

# ================================
# 2. CLEAN COLUMNS
# ================================
master_df.columns = (
    master_df.columns
    .str.strip()
    .str.upper()
    .str.replace(" ", "_")
)

# ================================
# 3. PIVOT TABLE
# ================================
pivot_df = master_df.pivot_table(
    index=['CIRCLE_NAME', 'LAC_ID', 'CELL_ID'],
    columns='PERIOD',
    values='SESSION_COUNT',
    aggfunc='sum'
).reset_index()

pivot_df = pivot_df.fillna(0)

baseline = "May 8_10"
mid = "May 15_17"
current = "May 22_24"

# ================================
# 4. CALCULATIONS
# ================================
pivot_df["ROLLING_AVG"] = pivot_df[[baseline, mid]].mean(axis=1)
pivot_df["ROLLING_AVG"] = pivot_df["ROLLING_AVG"].replace(0, 1)

pivot_df["PCT_CHANGE_ROLLING"] = (
    (pivot_df[current] - pivot_df["ROLLING_AVG"])
    / pivot_df["ROLLING_AVG"]
) * 100

pivot_df["LOSS"] = pivot_df["ROLLING_AVG"] - pivot_df[current]

# ================================
# 5. FLAG
# ================================
def flag(x):
    if x <= -70:
        return "STRONG_DROP"
    elif x <= -40:
        return "MODERATE_DROP"
    else:
        return "NORMAL"

pivot_df["FLAG"] = pivot_df["PCT_CHANGE_ROLLING"].apply(flag)

# ================================
# 6. LAT-LONG (SIMULATED)
# ================================
np.random.seed(42)

coords_df = pivot_df[['CELL_ID', 'LAC_ID']].copy()
coords_df["LAT"] = np.random.uniform(28.40, 28.90, len(coords_df))
coords_df["LONG"] = np.random.uniform(76.90, 77.40, len(coords_df))

pivot_df = pivot_df.merge(coords_df, on=["CELL_ID", "LAC_ID"], how="left")

# ================================
# 7. NEIGHBOR ANALYSIS
# ================================
coords = pivot_df[["LAT", "LONG"]]

nn = NearestNeighbors(n_neighbors=6)
nn.fit(coords)

distances, indices = nn.kneighbors(coords)

neighbor_data = []

for i in range(len(pivot_df)):
    for j in range(1, 6):
        neighbor_data.append({
            "CELL_ID": pivot_df.iloc[i]["CELL_ID"],
            "NEIGHBOR_CELL": pivot_df.iloc[indices[i][j]]["CELL_ID"],
            "DISTANCE": distances[i][j]
        })

neighbor_df = pd.DataFrame(neighbor_data)

# ================================
# 8. STATUS MERGE
# ================================
status_df = pivot_df[["CELL_ID", "FLAG"]].copy()
status_df = status_df.rename(columns={"FLAG": "CELL_STATUS"})

merged = neighbor_df.merge(status_df, on="CELL_ID", how="left")

merged = merged.merge(
    status_df,
    left_on="NEIGHBOR_CELL",
    right_on="CELL_ID",
    how="left"
)

merged = merged.rename(columns={
    "CELL_STATUS_x": "CELL_STATUS",
    "CELL_STATUS_y": "NEIGHBOR_STATUS"
})

merged = merged.drop(columns=["CELL_ID_y"])
merged = merged.rename(columns={"CELL_ID_x": "CELL_ID"})

# ================================
# 9. STRONG EVIDENCE
# ================================
merged["STRONG_EVIDENCE"] = (
    (merged["CELL_STATUS"] == "STRONG_DROP") &
    (merged["NEIGHBOR_STATUS"] != "STRONG_DROP")
)

# ================================
# 10. MERGE BACK
# ================================
final_table = pivot_df.merge(
    merged[["CELL_ID", "NEIGHBOR_CELL", "DISTANCE", "STRONG_EVIDENCE"]],
    on="CELL_ID",
    how="left"
)

# ================================
# 11. PRIORITY
# ================================
def priority(loss):
    if loss >= 20000:
        return "CRITICAL"
    elif loss >= 5000:
        return "HIGH"
    elif loss >= 1000:
        return "MEDIUM"
    else:
        return "LOW"

final_table["PRIORITY"] = final_table["LOSS"].apply(priority)

# ================================
# 12. IMPACT SCORE
# ================================
final_table["IMPACT_SCORE"] = (
    abs(final_table["PCT_CHANGE_ROLLING"]) * final_table["LOSS"]
)

# ================================
# 13. BEST NEIGHBOR
# ================================
final_table = final_table.sort_values(
    by=["CELL_ID", "STRONG_EVIDENCE", "DISTANCE"],
    ascending=[True, False, True]
)

final_unique = final_table.drop_duplicates(subset=["CELL_ID"]).copy()

# ================================
# BUSINESS IMPACT TIER
# ================================
def impact_tier(score):
    if score >= 1_000_000:
        return "High"
    elif score >= 300_000:
        return "Medium"
    else:
        return "Low"

final_unique["BUSINESS_IMPACT"] = final_unique["IMPACT_SCORE"].apply(impact_tier)

# ================================
# 16. LOSS SHARE
# ================================
final_unique["LOSS_SHARE_%"] = (
    final_unique["LOSS"] / final_unique["LOSS"].sum()
) * 100

# ================================
# 16.1 CONFIDENCE COMPONENTS
# ================================
final_unique["SESSION_SEVERITY"] = abs(final_unique["PCT_CHANGE_ROLLING"]).clip(0, 100)

final_unique["SIM_SEVERITY"] = (
    (final_unique["LOSS"] / final_unique["LOSS"].max()) * 100
)

final_unique["LOW_TRAFFIC_FLAG"] = np.where(final_unique["May 22_24"] < 1000, 100, 0)
final_unique["NEIGHBOR_SCORE"] = np.where(final_unique["STRONG_EVIDENCE"], 100, 0)

final_unique["PERSISTENCE"] = np.where(
    (final_unique["May 15_17"] < final_unique["May 8_10"]), 100, 50
)

# ================================
# 16.2 CONFIDENCE SCORE
# ================================
final_unique["CONFIDENCE_SCORE"] = (
    0.35 * final_unique["SIM_SEVERITY"] +
    0.35 * final_unique["SESSION_SEVERITY"] +
    0.15 * final_unique["LOW_TRAFFIC_FLAG"] +
    0.10 * final_unique["NEIGHBOR_SCORE"] +
    0.05 * final_unique["PERSISTENCE"]
)

def confidence_label(score):
    if score >= 80:
        return "High Confidence"
    elif score >= 60:
        return "Moderate"
    elif score >= 40:
        return "Watchlist"
    else:
        return "Low"

final_unique["CONFIDENCE_LEVEL"] = final_unique["CONFIDENCE_SCORE"].apply(confidence_label)

# ================================
# 16.4 RECOVERY RATIO
# ================================
final_unique["NEIGHBOR_GAIN"] = np.where(
    final_unique["STRONG_EVIDENCE"],
    final_unique["LOSS"] * 0.6,
    final_unique["LOSS"] * 0.2
)

final_unique["RECOVERY_RATIO"] = (
    final_unique["NEIGHBOR_GAIN"] / final_unique["LOSS"]
) * 100

def recovery_label(x):
    if x < 20:
        return "Low Migration"
    elif x < 70:
        return "Partial Migration"
    elif x <= 120:
        return "Strong Migration"
    else:
        return "Cluster/External Impact"

final_unique["RECOVERY_TYPE"] = final_unique["RECOVERY_RATIO"].apply(recovery_label)

# ================================
# FINAL STATUS (NOW CORRECT ORDER)
# ================================
def final_status(row):
    drop = row["PCT_CHANGE_ROLLING"]
    strong = row["STRONG_EVIDENCE"]
    current_traffic = row["May 22_24"]
    recovery = row["RECOVERY_RATIO"]

    if (drop <= -80) and (current_traffic < 1000) and strong:
        return "Suspected Inactive"

    elif (drop <= -70) and strong and (recovery >= 50):
        return "Possible Local Shift"

    elif (drop <= -70) and not strong:
        return "Area-wide Decline"

    elif (drop <= -40):
        return "Monitor"

    else:
        return "Healthy"

final_unique["FINAL_STATUS"] = final_unique.apply(final_status, axis=1)

# ================================
# ACTION
# ================================
def assign_action(status):
    if status == "Healthy":
        return "No action"
    elif status == "Monitor":
        return "Observe next period"
    elif status == "Possible Local Shift":
        return "Review neighbor load / balancing"
    elif status == "Suspected Inactive":
        return "Verify site and cell availability"
    elif status == "Area-wide Decline":
        return "Review LAC-level network issue"
    else:
        return "No action"

final_unique["ACTION"] = final_unique["FINAL_STATUS"].apply(assign_action)

# ================================
# SORT
# ================================
priority_order = {
    "CRITICAL": 1,
    "HIGH": 2,
    "MEDIUM": 3,
    "LOW": 4
}

final_unique["PRIORITY_RANK"] = final_unique["PRIORITY"].map(priority_order)

final_unique = final_unique.sort_values(
    by=["PRIORITY_RANK", "IMPACT_SCORE"],
    ascending=[True, False]
)

# ================================
# EXPORT
# ================================
final_unique.to_csv("FINAL_TELECOM_DASHBOARD.csv", index=False)

# ================================
# SUMMARY
# ================================
print("\n📊 TOTAL CELLS:", final_unique["CELL_ID"].nunique())

print("\n📊 STATUS COUNT:")
print(final_unique["FINAL_STATUS"].value_counts())

print("\n TOP 10 ISSUES:")
print(final_unique.head(10))

In [ ]:
import pandas as pd
import numpy as np
from sklearn.neighbors import NearestNeighbors

# ================================
# 1. LOAD DATA
# ================================
file_path = r"C:\Users\malik\Downloads\LAC_CELL_CIRCLE_2G_SITES_DATA_FINAL.xlsx"

df1 = pd.read_excel(file_path, sheet_name=0)
df2 = pd.read_excel(file_path, sheet_name=1)
df3 = pd.read_excel(file_path, sheet_name=2)

df1["PERIOD"] = "May 8_10"
df2["PERIOD"] = "May 15_17"
df3["PERIOD"] = "May 22_24"

master_df = pd.concat([df1, df2, df3], ignore_index=True)

# ================================
# 2. CLEAN
# ================================
master_df.columns = (
    master_df.columns
    .str.strip()
    .str.upper()
    .str.replace(" ", "_")
)

# ================================
# 3. PIVOT
# ================================
pivot_df = master_df.pivot_table(
    index=['CIRCLE_NAME', 'LAC_ID', 'CELL_ID'],
    columns='PERIOD',
    values='SESSION_COUNT',
    aggfunc='sum'
).reset_index().fillna(0)

baseline = "May 8_10"
mid = "May 15_17"
current = "May 22_24"

# ================================
# 4. CORE METRICS
# ================================
pivot_df["ROLLING_AVG"] = pivot_df[[baseline, mid]].mean(axis=1).replace(0, 1)

pivot_df["PCT_CHANGE_ROLLING"] = (
    (pivot_df[current] - pivot_df["ROLLING_AVG"]) / pivot_df["ROLLING_AVG"]
) * 100

pivot_df["LOSS"] = pivot_df["ROLLING_AVG"] - pivot_df[current]

# ================================
# 5. TREND CONSISTENCY (NEW 🔥)
# ================================
pivot_df["TREND_DROP"] = np.where(
    (pivot_df[current] < pivot_df[mid]) &
    (pivot_df[mid] < pivot_df[baseline]),
    1, 0
)

# ================================
# 6. FLAG
# ================================
def flag(x):
    if x <= -70:
        return "STRONG_DROP"
    elif x <= -40:
        return "MODERATE_DROP"
    else:
        return "NORMAL"

pivot_df["FLAG"] = pivot_df["PCT_CHANGE_ROLLING"].apply(flag)

# ================================
# 7. LAC LEVEL IMPACT (NEW 🔥)
# ================================
lac_impact = pivot_df.groupby("LAC_ID")["PCT_CHANGE_ROLLING"].mean().reset_index()
lac_impact.rename(columns={"PCT_CHANGE_ROLLING": "LAC_IMPACT"}, inplace=True)

pivot_df = pivot_df.merge(lac_impact, on="LAC_ID", how="left")

# ================================
# 8. LAT LONG (KEEP / REPLACE IF REAL DATA)
# ================================
np.random.seed(42)

pivot_df["LAT"] = np.random.uniform(28.40, 28.90, len(pivot_df))
pivot_df["LONG"] = np.random.uniform(76.90, 77.40, len(pivot_df))

# ================================
# 9. NEIGHBOR ANALYSIS
# ================================
coords = pivot_df[["LAT", "LONG"]]

nn = NearestNeighbors(n_neighbors=6)
nn.fit(coords)

distances, indices = nn.kneighbors(coords)

neighbor_data = []

for i in range(len(pivot_df)):
    for j in range(1, 6):
        neighbor_data.append({
            "CELL_ID": pivot_df.iloc[i]["CELL_ID"],
            "NEIGHBOR_CELL": pivot_df.iloc[indices[i][j]]["CELL_ID"],
            "DISTANCE": distances[i][j]
        })

neighbor_df = pd.DataFrame(neighbor_data)

# ================================
# 10. STATUS MERGE
# ================================
status_df = pivot_df[["CELL_ID", "FLAG"]].rename(columns={"FLAG": "CELL_STATUS"})

merged = neighbor_df.merge(status_df, on="CELL_ID")
merged = merged.merge(
    status_df,
    left_on="NEIGHBOR_CELL",
    right_on="CELL_ID",
    suffixes=("", "_NEIGHBOR")
)

merged.rename(columns={"CELL_STATUS_NEIGHBOR": "NEIGHBOR_STATUS"}, inplace=True)

# ================================
# 11. STRONG EVIDENCE
# ================================
merged["STRONG_EVIDENCE"] = (
    (merged["CELL_STATUS"] == "STRONG_DROP") &
    (merged["NEIGHBOR_STATUS"] != "STRONG_DROP")
)

# ================================
# 12. MERGE BACK
# ================================
final_table = pivot_df.merge(
    merged[["CELL_ID", "NEIGHBOR_CELL", "DISTANCE", "STRONG_EVIDENCE"]],
    on="CELL_ID",
    how="left"
)

# ================================
# 13. PRIORITY (IMPROVED 🔥)
# ================================
def priority(row):
    if row["LOSS"] > 20000 and row["PCT_CHANGE_ROLLING"] < -70:
        return "CRITICAL"
    elif row["LOSS"] > 5000:
        return "HIGH"
    elif row["LOSS"] > 1000:
        return "MEDIUM"
    else:
        return "LOW"

final_table["PRIORITY"] = final_table.apply(priority, axis=1)

# ================================
# 14. IMPACT SCORE
# ================================
final_table["IMPACT_SCORE"] = (
    abs(final_table["PCT_CHANGE_ROLLING"]) * final_table["LOSS"]
)

# ================================
# 15. BEST NEIGHBOR
# ================================
final_table = final_table.sort_values(
    by=["CELL_ID", "STRONG_EVIDENCE", "DISTANCE"],
    ascending=[True, False, True]
)

final_unique = final_table.drop_duplicates("CELL_ID").copy()

# ================================
# 16. CONFIDENCE
# ================================
final_unique["SESSION_SEVERITY"] = abs(final_unique["PCT_CHANGE_ROLLING"]).clip(0, 100)

final_unique["SIM_SEVERITY"] = (
    (final_unique["LOSS"] / final_unique["LOSS"].max()) * 100
)

final_unique["TREND_SCORE"] = final_unique["TREND_DROP"] * 100
final_unique["NEIGHBOR_SCORE"] = np.where(final_unique["STRONG_EVIDENCE"], 100, 0)

final_unique["CONFIDENCE_SCORE"] = (
    0.30 * final_unique["SIM_SEVERITY"] +
    0.30 * final_unique["SESSION_SEVERITY"] +
    0.20 * final_unique["TREND_SCORE"] +
    0.20 * final_unique["NEIGHBOR_SCORE"]
)

def confidence_label(score):
    if score >= 80:
        return "High Confidence"
    elif score >= 60:
        return "Moderate"
    elif score >= 40:
        return "Watchlist"
    else:
        return "Low"

final_unique["CONFIDENCE_LEVEL"] = final_unique["CONFIDENCE_SCORE"].apply(confidence_label)

# ================================
# 17. FINAL STATUS (BEST LOGIC 🔥)
# ================================
def final_status(row):
    drop = row["PCT_CHANGE_ROLLING"]
    strong = row["STRONG_EVIDENCE"]
    lac = row["LAC_IMPACT"]
    trend = row["TREND_DROP"]

    if (drop <= -80) and strong and trend:
        return "Suspected Inactive"

    elif (drop <= -70) and strong:
        return "Possible Local Shift"

    elif (lac <= -50):
        return "Area-wide Decline"

    elif (drop <= -40):
        return "Monitor"

    else:
        return "Healthy"

final_unique["FINAL_STATUS"] = final_unique.apply(final_status, axis=1)

# ================================
# 18. ACTION
# ================================
action_map = {
    "Healthy": "No action",
    "Monitor": "Observe next period",
    "Possible Local Shift": "Review neighbor load / balancing",
    "Suspected Inactive": "Verify site and cell availability",
    "Area-wide Decline": "Check LAC / network issue"
}

final_unique["ACTION"] = final_unique["FINAL_STATUS"].map(action_map)

# ================================
# 19. SORT
# ================================
priority_order = {"CRITICAL": 1, "HIGH": 2, "MEDIUM": 3, "LOW": 4}

final_unique["PRIORITY_RANK"] = final_unique["PRIORITY"].map(priority_order)

final_unique = final_unique.sort_values(
    by=["PRIORITY_RANK", "IMPACT_SCORE"],
    ascending=[True, False]
)

# ================================
# 20. EXPORT
# ================================
final_unique.to_csv("FINAL_TELECOM_DASHBOARD_V2.csv", index=False)

# ================================
# 21. SUMMARY
# ================================
print("\n📊 TOTAL CELLS:", final_unique["CELL_ID"].nunique())

print("\n📊 STATUS COUNT:")
print(final_unique["FINAL_STATUS"].value_counts())

print("\n🔥 TOP 10 ISSUES:")
print(final_unique.head(10))

In [ ]:
import pandas as pd
import numpy as np
from sklearn.neighbors import NearestNeighbors

# ================================
# 1. LOAD DATA
# ================================
file_path = r"C:\Users\malik\Downloads\LAC_CELL_CIRCLE_2G_SITES_DATA_FINAL.xlsx"

df1 = pd.read_excel(file_path, sheet_name=0)
df2 = pd.read_excel(file_path, sheet_name=1)
df3 = pd.read_excel(file_path, sheet_name=2)

df1["PERIOD"] = "May 8_10"
df2["PERIOD"] = "May 15_17"
df3["PERIOD"] = "May 22_24"

master_df = pd.concat([df1, df2, df3], ignore_index=True)

# ================================
# 2. CLEAN
# ================================
master_df.columns = (
    master_df.columns
    .str.strip()
    .str.upper()
    .str.replace(" ", "_")
)

# ================================
# 3. PIVOT
# ================================
pivot_df = master_df.pivot_table(
    index=['CIRCLE_NAME', 'LAC_ID', 'CELL_ID'],
    columns='PERIOD',
    values='SESSION_COUNT',
    aggfunc='sum'
).reset_index().fillna(0)

baseline = "May 8_10"
mid = "May 15_17"
current = "May 22_24"

# ================================
# 4. CORE METRICS
# ================================
pivot_df["ROLLING_AVG"] = pivot_df[[baseline, mid]].mean(axis=1).replace(0, 1)

pivot_df["PCT_CHANGE_ROLLING"] = (
    (pivot_df[current] - pivot_df["ROLLING_AVG"]) / pivot_df["ROLLING_AVG"]
) * 100

pivot_df["LOSS"] = pivot_df["ROLLING_AVG"] - pivot_df[current]

# ================================
# 5. TREND
# ================================
pivot_df["TREND_DROP"] = np.where(
    (pivot_df[current] < pivot_df[mid]) &
    (pivot_df[mid] < pivot_df[baseline]),
    1, 0
)

# ================================
# 6. LAC IMPACT
# ================================
lac_impact = pivot_df.groupby("LAC_ID")["PCT_CHANGE_ROLLING"].mean().reset_index()
lac_impact.rename(columns={"PCT_CHANGE_ROLLING": "LAC_IMPACT"}, inplace=True)
pivot_df = pivot_df.merge(lac_impact, on="LAC_ID", how="left")

# ================================
# 7. LAT LONG (DUMMY)
# ================================
np.random.seed(42)
pivot_df["LAT"] = np.random.uniform(28.40, 28.90, len(pivot_df))
pivot_df["LONG"] = np.random.uniform(76.90, 77.40, len(pivot_df))

# ================================
# 8. NEIGHBOR ANALYSIS
# ================================
coords = pivot_df[["LAT", "LONG"]]
nn = NearestNeighbors(n_neighbors=6)
nn.fit(coords)

distances, indices = nn.kneighbors(coords)

neighbor_data = []
for i in range(len(pivot_df)):
    for j in range(1, 6):
        neighbor_data.append({
            "CELL_ID": pivot_df.iloc[i]["CELL_ID"],
            "NEIGHBOR_CELL": pivot_df.iloc[indices[i][j]]["CELL_ID"],
            "DISTANCE": distances[i][j]
        })

neighbor_df = pd.DataFrame(neighbor_data)

# ================================
# 9. NEIGHBOR GAIN
# ================================
neighbor_values = pivot_df[["CELL_ID", current]].rename(
    columns={"CELL_ID": "NEIGHBOR_CELL", current: "NEIGHBOR_CURRENT"}
)

cell_values = pivot_df[["CELL_ID", current]].rename(
    columns={current: "CELL_CURRENT"}
)

merged = neighbor_df.merge(cell_values, on="CELL_ID")
merged = merged.merge(neighbor_values, on="NEIGHBOR_CELL")

merged["NEIGHBOR_GAIN"] = merged["NEIGHBOR_CURRENT"] - merged["CELL_CURRENT"]
merged["POSITIVE_GAIN"] = np.where(merged["NEIGHBOR_GAIN"] > 0, merged["NEIGHBOR_GAIN"], 0)

# Aggregate
neighbor_agg = merged.groupby("CELL_ID").agg({
    "POSITIVE_GAIN": "sum",
    "DISTANCE": "mean"
}).reset_index()

neighbor_agg.rename(columns={
    "POSITIVE_GAIN": "TOTAL_NEIGHBOR_GAIN",
    "DISTANCE": "AVG_NEIGHBOR_DISTANCE"
}, inplace=True)

final_table = pivot_df.merge(neighbor_agg, on="CELL_ID", how="left")

# ================================
# 10. RECOVERY RATIO 🔥
# ================================
final_table["RECOVERY_RATIO"] = (
    final_table["TOTAL_NEIGHBOR_GAIN"] / final_table["LOSS"].replace(0, 1)
)

# ================================
# 11. PRIORITY
# ================================
def priority(row):
    if row["LOSS"] > 20000 and row["PCT_CHANGE_ROLLING"] < -70:
        return "CRITICAL"
    elif row["LOSS"] > 5000:
        return "HIGH"
    elif row["LOSS"] > 1000:
        return "MEDIUM"
    else:
        return "LOW"

final_table["PRIORITY"] = final_table.apply(priority, axis=1)

# ================================
# 12. IMPACT SCORE
# ================================
final_table["IMPACT_SCORE"] = (
    abs(final_table["PCT_CHANGE_ROLLING"]) * final_table["LOSS"]
)

# ================================
# 13. UNIQUE CELL
# ================================
final_unique = final_table.copy()

# ================================
# 14. CONFIDENCE
# ================================
final_unique["SESSION_SEVERITY"] = abs(final_unique["PCT_CHANGE_ROLLING"]).clip(0, 100)

final_unique["SIM_SEVERITY"] = (
    (final_unique["LOSS"] / final_unique["LOSS"].max()) * 100
)

final_unique["TREND_SCORE"] = final_unique["TREND_DROP"] * 100
final_unique["NEIGHBOR_SCORE"] = np.clip(final_unique["RECOVERY_RATIO"] * 100, 0, 100)

trend_penalty = np.where(final_unique["TREND_DROP"] == 0, -10, 0)

mismatch_penalty = np.where(
    abs(final_unique["SESSION_SEVERITY"] - final_unique["SIM_SEVERITY"]) > 40,
    -10, 0
)

final_unique["CONFIDENCE_SCORE"] = (
    0.30 * final_unique["SIM_SEVERITY"] +
    0.30 * final_unique["SESSION_SEVERITY"] +
    0.20 * final_unique["TREND_SCORE"] +
    0.20 * final_unique["NEIGHBOR_SCORE"] +
    trend_penalty + mismatch_penalty
)

def confidence_label(score):
    if score >= 80:
        return "High Confidence"
    elif score >= 60:
        return "Moderate"
    elif score >= 40:
        return "Watchlist"
    else:
        return "Low"

final_unique["CONFIDENCE_LEVEL"] = final_unique["CONFIDENCE_SCORE"].apply(confidence_label)

# ================================
# 15. FINAL STATUS
# ================================
def final_status(row):
    drop = row["PCT_CHANGE_ROLLING"]
    recovery = row["RECOVERY_RATIO"]
    lac = row["LAC_IMPACT"]
    trend = row["TREND_DROP"]

    if (drop <= -85) and (recovery < 0.3) and trend:
        return "Suspected Inactive"

    elif (drop <= -70) and (recovery >= 0.3):
        return "Possible Local Shift"

    elif (lac <= -50) and (recovery < 0.2):
        return "Area-wide Decline"

    elif (drop <= -40):
        return "Monitor"

    else:
        return "Healthy"

final_unique["FINAL_STATUS"] = final_unique.apply(final_status, axis=1)

# ================================
# 16. ACTION
# ================================
action_map = {
    "Healthy": "No action",
    "Monitor": "Observe next period",
    "Possible Local Shift": "Review neighbor load / balancing",
    "Suspected Inactive": "Verify site and cell availability",
    "Area-wide Decline": "Check LAC / network issue"
}

final_unique["ACTION"] = final_unique["FINAL_STATUS"].map(action_map)

# ================================
# 17. BUSINESS IMPACT
# ================================
def impact_tier(x):
    if x > 2000000:
        return "High"
    elif x > 500000:
        return "Medium"
    else:
        return "Low"

final_unique["BUSINESS_IMPACT"] = final_unique["IMPACT_SCORE"].apply(impact_tier)

# ================================
# 18. PRIORITY LABEL
# ================================
priority_map = {
    "CRITICAL": "P1 - Critical",
    "HIGH": "P2 - High",
    "MEDIUM": "P3 - Medium",
    "LOW": "P4 - Low"
}

final_unique["PRIORITY_LABEL"] = final_unique["PRIORITY"].map(priority_map)

priority_order = {"CRITICAL": 1, "HIGH": 2, "MEDIUM": 3, "LOW": 4}
final_unique["PRIORITY_RANK"] = final_unique["PRIORITY"].map(priority_order)

# ================================
# 19. SORT
# ================================
final_unique = final_unique.sort_values(
    by=["PRIORITY_RANK", "IMPACT_SCORE", "CONFIDENCE_SCORE"],
    ascending=[True, False, False]
)

# ================================
# 20. EXPORT
# ================================
final_unique.to_csv("FINAL_TELECOM_DASHBOARD_V3.csv", index=False)

# ================================
# 21. SUMMARY
# ================================
print("\n📊 TOTAL CELLS:", final_unique["CELL_ID"].nunique())

print("\n📊 STATUS COUNT:")
print(final_unique["FINAL_STATUS"].value_counts())

print("\n🔥 TOP 10 ISSUES:")
print(final_unique.head(10))

In [ ]:
import pandas as pd
import numpy as np
from sklearn.neighbors import NearestNeighbors

# ================================
# 1. LOAD DATA
# ================================
file_path = r"C:\Users\malik\Downloads\LAC_CELL_CIRCLE_2G_SITES_DATA_FINAL.xlsx"

df1 = pd.read_excel(file_path, sheet_name=0)
df2 = pd.read_excel(file_path, sheet_name=1)
df3 = pd.read_excel(file_path, sheet_name=2)

df1["PERIOD"] = "May 8_10"
df2["PERIOD"] = "May 15_17"
df3["PERIOD"] = "May 22_24"

master_df = pd.concat([df1, df2, df3], ignore_index=True)

# ================================
# 2. CLEAN
# ================================
master_df.columns = (
    master_df.columns
    .str.strip()
    .str.upper()
    .str.replace(" ", "_")
)

# ================================
# 3. PIVOT
# ================================
pivot_df = master_df.pivot_table(
    index=['CIRCLE_NAME', 'LAC_ID', 'CELL_ID'],
    columns='PERIOD',
    values='SESSION_COUNT',
    aggfunc='sum'
).reset_index().fillna(0)

baseline = "May 8_10"
mid = "May 15_17"
current = "May 22_24"

# ================================
# 4. CORE METRICS
# ================================
pivot_df["ROLLING_AVG"] = pivot_df[[baseline, mid]].mean(axis=1).replace(0, 1)

pivot_df["PCT_CHANGE_ROLLING"] = (
    (pivot_df[current] - pivot_df["ROLLING_AVG"]) / pivot_df["ROLLING_AVG"]
) * 100

pivot_df["LOSS"] = pivot_df["ROLLING_AVG"] - pivot_df[current]

# ================================
# 5. TREND
# ================================
pivot_df["TREND_DROP"] = np.where(
    (pivot_df[current] < pivot_df[mid]) &
    (pivot_df[mid] < pivot_df[baseline]),
    1, 0
)

# ================================
# 6. LAC IMPACT
# ================================
lac_impact = pivot_df.groupby("LAC_ID")["PCT_CHANGE_ROLLING"].mean().reset_index()
lac_impact.rename(columns={"PCT_CHANGE_ROLLING": "LAC_IMPACT"}, inplace=True)
pivot_df = pivot_df.merge(lac_impact, on="LAC_ID", how="left")

# ================================
# 7. LAT LONG (DUMMY)
# ================================
np.random.seed(42)
pivot_df["LAT"] = np.random.uniform(28.40, 28.90, len(pivot_df))
pivot_df["LONG"] = np.random.uniform(76.90, 77.40, len(pivot_df))

# ================================
# 8. NEIGHBOR ANALYSIS
# ================================
coords = pivot_df[["LAT", "LONG"]]
nn = NearestNeighbors(n_neighbors=6)
nn.fit(coords)

distances, indices = nn.kneighbors(coords)

neighbor_data = []
for i in range(len(pivot_df)):
    for j in range(1, 6):
        neighbor_data.append({
            "CELL_ID": pivot_df.iloc[i]["CELL_ID"],
            "NEIGHBOR_CELL": pivot_df.iloc[indices[i][j]]["CELL_ID"],
            "DISTANCE": distances[i][j]
        })

neighbor_df = pd.DataFrame(neighbor_data)

# ================================
# 9. NEIGHBOR GAIN
# ================================
neighbor_values = pivot_df[["CELL_ID", current]].rename(
    columns={"CELL_ID": "NEIGHBOR_CELL", current: "NEIGHBOR_CURRENT"}
)

cell_values = pivot_df[["CELL_ID", current]].rename(
    columns={current: "CELL_CURRENT"}
)

merged = neighbor_df.merge(cell_values, on="CELL_ID")
merged = merged.merge(neighbor_values, on="NEIGHBOR_CELL")

merged["NEIGHBOR_GAIN"] = merged["NEIGHBOR_CURRENT"] - merged["CELL_CURRENT"]
merged["POSITIVE_GAIN"] = np.where(merged["NEIGHBOR_GAIN"] > 0, merged["NEIGHBOR_GAIN"], 0)

neighbor_agg = merged.groupby("CELL_ID").agg({
    "POSITIVE_GAIN": "sum",
    "DISTANCE": "mean"
}).reset_index()

neighbor_agg.rename(columns={
    "POSITIVE_GAIN": "TOTAL_NEIGHBOR_GAIN",
    "DISTANCE": "AVG_NEIGHBOR_DISTANCE"
}, inplace=True)

final_table = pivot_df.merge(neighbor_agg, on="CELL_ID", how="left")

# ================================
# 10. RECOVERY RATIO
# ================================
final_table["RECOVERY_RATIO"] = (
    final_table["TOTAL_NEIGHBOR_GAIN"] / final_table["LOSS"].replace(0, 1)
)

# ================================
# 11. PRIORITY
# ================================
def priority(row):
    if row["LOSS"] > 20000 and row["PCT_CHANGE_ROLLING"] < -70:
        return "CRITICAL"
    elif row["LOSS"] > 5000:
        return "HIGH"
    elif row["LOSS"] > 1000:
        return "MEDIUM"
    else:
        return "LOW"

final_table["PRIORITY"] = final_table.apply(priority, axis=1)

# ================================
# 12. IMPACT SCORE
# ================================
final_table["IMPACT_SCORE"] = (
    abs(final_table["PCT_CHANGE_ROLLING"]) * final_table["LOSS"]
)

# ================================
# 13. UNIQUE CELL FIX ✅
# ================================
final_unique = final_table.drop_duplicates(subset=["CELL_ID"])

# ================================
# 14. CONFIDENCE
# ================================
final_unique["SESSION_SEVERITY"] = abs(final_unique["PCT_CHANGE_ROLLING"]).clip(0, 100)

final_unique["SIM_SEVERITY"] = (
    (final_unique["LOSS"] / final_unique["LOSS"].max()) * 100
)

final_unique["TREND_SCORE"] = final_unique["TREND_DROP"] * 100
final_unique["NEIGHBOR_SCORE"] = np.clip(final_unique["RECOVERY_RATIO"] * 100, 0, 100)

trend_penalty = np.where(final_unique["TREND_DROP"] == 0, -10, 0)

mismatch_penalty = np.where(
    abs(final_unique["SESSION_SEVERITY"] - final_unique["SIM_SEVERITY"]) > 40,
    -10, 0
)

final_unique["CONFIDENCE_SCORE"] = (
    0.30 * final_unique["SIM_SEVERITY"] +
    0.30 * final_unique["SESSION_SEVERITY"] +
    0.20 * final_unique["TREND_SCORE"] +
    0.20 * final_unique["NEIGHBOR_SCORE"] +
    trend_penalty + mismatch_penalty
)

def confidence_label(score):
    if score >= 80:
        return "High Confidence"
    elif score >= 60:
        return "Moderate"
    elif score >= 40:
        return "Watchlist"
    else:
        return "Low"

final_unique["CONFIDENCE_LEVEL"] = final_unique["CONFIDENCE_SCORE"].apply(confidence_label)

# ================================
# 15. FINAL STATUS (UPGRADED 🚀)
# ================================
def final_status(row):
    drop = row["PCT_CHANGE_ROLLING"]
    recovery = row["RECOVERY_RATIO"]
    lac = row["LAC_IMPACT"]
    trend = row["TREND_DROP"]
    loss = row["LOSS"]

    if (drop <= -85) and (recovery < 0.3) and trend:
        return "Suspected Inactive"

    elif (loss > 50000) and (drop < -20):
        return "High Value Degradation"

    elif (lac <= -40) and (drop <= -40):
        return "Cluster Degradation"

    elif (drop <= -70) and (recovery >= 0.3):
        return "Possible Local Shift"

    elif (lac <= -50) and (recovery < 0.2):
        return "Area-wide Decline"

    elif (drop <= -40):
        return "Monitor"

    else:
        return "Healthy"

final_unique["FINAL_STATUS"] = final_unique.apply(final_status, axis=1)

# ================================
# 16. ACTION
# ================================
action_map = {
    "Healthy": "No action",
    "Monitor": "Observe next period",
    "Possible Local Shift": "Review neighbor load / balancing",
    "Suspected Inactive": "Verify site and cell availability",
    "Area-wide Decline": "Check LAC / network issue",
    "High Value Degradation": "Urgent optimization required",
    "Cluster Degradation": "Investigate cluster performance"
}

final_unique["ACTION"] = final_unique["FINAL_STATUS"].map(action_map)

# ================================
# 17. BUSINESS IMPACT
# ================================
def impact_tier(x):
    if x > 2000000:
        return "High"
    elif x > 500000:
        return "Medium"
    else:
        return "Low"

final_unique["BUSINESS_IMPACT"] = final_unique["IMPACT_SCORE"].apply(impact_tier)

# ================================
# 18. PRIORITY LABEL
# ================================
priority_map = {
    "CRITICAL": "P1 - Critical",
    "HIGH": "P2 - High",
    "MEDIUM": "P3 - Medium",
    "LOW": "P4 - Low"
}

final_unique["PRIORITY_LABEL"] = final_unique["PRIORITY"].map(priority_map)

priority_order = {"CRITICAL": 1, "HIGH": 2, "MEDIUM": 3, "LOW": 4}
final_unique["PRIORITY_RANK"] = final_unique["PRIORITY"].map(priority_order)

# ================================
# 19. SORT
# ================================
final_unique = final_unique.sort_values(
    by=["PRIORITY_RANK", "IMPACT_SCORE", "CONFIDENCE_SCORE"],
    ascending=[True, False, False]
)

# ================================
# 20. EXPORT
# ================================
final_unique.to_csv("FINAL_TELECOM_DASHBOARD_V4.csv", index=False)

# ================================
# 21. SUMMARY
# ================================
print("\n📊 TOTAL UNIQUE CELLS:", final_unique["CELL_ID"].nunique())

print("\n📊 STATUS COUNT:")
print(final_unique["FINAL_STATUS"].value_counts())

print("\n🔥 TOP 10 ISSUES:")
print(final_unique.head(10))

In [ ]:
import pandas as pd
import numpy as np
from sklearn.neighbors import NearestNeighbors

# ================================
# 1. LOAD DATA
# ================================
file_path = r"C:\Users\malik\Downloads\LAC_CELL_CIRCLE_2G_SITES_DATA_FINAL.xlsx"

df1 = pd.read_excel(file_path, sheet_name=0)
df2 = pd.read_excel(file_path, sheet_name=1)
df3 = pd.read_excel(file_path, sheet_name=2)

df1["PERIOD"] = "May 8_10"
df2["PERIOD"] = "May 15_17"
df3["PERIOD"] = "May 22_24"

master_df = pd.concat([df1, df2, df3], ignore_index=True)

# ================================
# 2. CLEAN
# ================================
master_df.columns = (
    master_df.columns
    .str.strip()
    .str.upper()
    .str.replace(" ", "_")
)

# ================================
# 3. PIVOT
# ================================
pivot_df = master_df.pivot_table(
    index=['CIRCLE_NAME', 'LAC_ID', 'CELL_ID'],
    columns='PERIOD',
    values='SESSION_COUNT',
    aggfunc='sum'
).reset_index().fillna(0)

baseline = "May 8_10"
mid = "May 15_17"
current = "May 22_24"

# ================================
# 4. CORE METRICS
# ================================
pivot_df["ROLLING_AVG"] = pivot_df[[baseline, mid]].mean(axis=1).replace(0, 1)

pivot_df["PCT_CHANGE_ROLLING"] = (
    (pivot_df[current] - pivot_df["ROLLING_AVG"]) / pivot_df["ROLLING_AVG"]
) * 100

pivot_df["LOSS"] = pivot_df["ROLLING_AVG"] - pivot_df[current]

# ================================
# 5. TREND
# ================================
pivot_df["TREND_DROP"] = np.where(
    (pivot_df[current] < pivot_df[mid]) &
    (pivot_df[mid] < pivot_df[baseline]),
    1, 0
)

# ================================
# 6. LAC IMPACT
# ================================
lac_impact = pivot_df.groupby("LAC_ID")["PCT_CHANGE_ROLLING"].mean().reset_index()
lac_impact.rename(columns={"PCT_CHANGE_ROLLING": "LAC_IMPACT"}, inplace=True)
pivot_df = pivot_df.merge(lac_impact, on="LAC_ID", how="left")

# ================================
# 7. LAT LONG (DUMMY)
# ================================
np.random.seed(42)
pivot_df["LAT"] = np.random.uniform(28.40, 28.90, len(pivot_df))
pivot_df["LONG"] = np.random.uniform(76.90, 77.40, len(pivot_df))

# ================================
# 8. NEIGHBOR ANALYSIS
# ================================
coords = pivot_df[["LAT", "LONG"]]
nn = NearestNeighbors(n_neighbors=6)
nn.fit(coords)

distances, indices = nn.kneighbors(coords)

neighbor_data = []
for i in range(len(pivot_df)):
    for j in range(1, 6):
        neighbor_data.append({
            "CELL_ID": pivot_df.iloc[i]["CELL_ID"],
            "NEIGHBOR_CELL": pivot_df.iloc[indices[i][j]]["CELL_ID"],
            "DISTANCE": distances[i][j]
        })

neighbor_df = pd.DataFrame(neighbor_data)

# ================================
# 9. NEIGHBOR GAIN
# ================================
neighbor_values = pivot_df[["CELL_ID", current]].rename(
    columns={"CELL_ID": "NEIGHBOR_CELL", current: "NEIGHBOR_CURRENT"}
)

cell_values = pivot_df[["CELL_ID", current]].rename(
    columns={current: "CELL_CURRENT"}
)

merged = neighbor_df.merge(cell_values, on="CELL_ID")
merged = merged.merge(neighbor_values, on="NEIGHBOR_CELL")

merged["NEIGHBOR_GAIN"] = merged["NEIGHBOR_CURRENT"] - merged["CELL_CURRENT"]
merged["POSITIVE_GAIN"] = np.where(merged["NEIGHBOR_GAIN"] > 0, merged["NEIGHBOR_GAIN"], 0)

neighbor_agg = merged.groupby("CELL_ID").agg({
    "POSITIVE_GAIN": "sum",
    "DISTANCE": "mean"
}).reset_index()

neighbor_agg.rename(columns={
    "POSITIVE_GAIN": "TOTAL_NEIGHBOR_GAIN",
    "DISTANCE": "AVG_NEIGHBOR_DISTANCE"
}, inplace=True)

final_table = pivot_df.merge(neighbor_agg, on="CELL_ID", how="left")

# ================================
# 10. RECOVERY RATIO
# ================================
final_table["LOSS_SAFE"] = final_table["LOSS"].replace(0, 1)

final_table["RECOVERY_RATIO"] = (
    final_table["TOTAL_NEIGHBOR_GAIN"] / final_table["LOSS_SAFE"]
).clip(0, 2)

# ================================
# 11. IMPACT SCORE (FIXED ORDER)
# ================================
final_table["IMPACT_SCORE"] = (
    abs(final_table["PCT_CHANGE_ROLLING"]) * final_table["LOSS"]
)

# ================================
# 12. UNIQUE CELL
# ================================
final_unique = final_table.drop_duplicates(subset=["CELL_ID"]).copy()

# ================================
# 13. CONFIDENCE
# ================================
final_unique["SESSION_SEVERITY"] = abs(final_unique["PCT_CHANGE_ROLLING"]).clip(0, 100)

final_unique["SIM_SEVERITY"] = (
    (final_unique["LOSS"] / final_unique["LOSS"].max()) * 100
)

final_unique["TREND_SCORE"] = final_unique["TREND_DROP"] * 100
final_unique["NEIGHBOR_SCORE"] = np.clip(final_unique["RECOVERY_RATIO"] * 100, 0, 100)

trend_penalty = np.where(final_unique["TREND_DROP"] == 0, -10, 0)
mismatch_penalty = np.where(
    abs(final_unique["SESSION_SEVERITY"] - final_unique["SIM_SEVERITY"]) > 40,
    -10, 0
)

final_unique["CONFIDENCE_SCORE"] = (
    0.30 * final_unique["SIM_SEVERITY"] +
    0.30 * final_unique["SESSION_SEVERITY"] +
    0.20 * final_unique["TREND_SCORE"] +
    0.20 * final_unique["NEIGHBOR_SCORE"] +
    trend_penalty + mismatch_penalty
).clip(0, 100)

def confidence_label(score):
    if score >= 80:
        return "High Confidence"
    elif score >= 60:
        return "Moderate"
    elif score >= 40:
        return "Watchlist"
    else:
        return "Low"

final_unique["CONFIDENCE_LEVEL"] = final_unique["CONFIDENCE_SCORE"].apply(confidence_label)

# ================================
# 14. PRIORITY (FIXED ✅)
# ================================
def priority(row):
    impact = row["IMPACT_SCORE"]
    confidence = row["CONFIDENCE_SCORE"]

    if (impact > 1500000 and confidence >= 70):
        return "CRITICAL"
    elif (impact > 500000 and confidence >= 60):
        return "HIGH"
    elif (impact > 200000 and confidence >= 40):
        return "MEDIUM"
    else:
        return "LOW"

final_unique["PRIORITY"] = final_unique.apply(priority, axis=1)

# ================================
# 15. FINAL STATUS
# ================================
def final_status(row):
    drop = row["PCT_CHANGE_ROLLING"]
    recovery = row["RECOVERY_RATIO"]
    lac = row["LAC_IMPACT"]
    trend = row["TREND_DROP"]
    loss = row["LOSS"]
    confidence = row["CONFIDENCE_SCORE"]
    current_val = row[current]
    baseline_val = row["ROLLING_AVG"]

    if (drop <= -85) and (current_val < 0.1 * baseline_val) and (confidence >= 70):
        return "Suspected Inactive"
    elif (drop <= -70) and (recovery >= 0.5) and (confidence >= 60):
        return "Possible Local Shift"
    elif (lac <= -40) and (recovery < 0.2):
        return "Area-wide Decline"
    elif (lac <= -25) and (drop <= -40):
        return "Local Area Degradation"
    elif (loss > 40000) and (drop <= -50):
        return "High Value Degradation"
    elif (drop <= -40):
        return "Monitor"
    else:
        return "Healthy"

final_unique["FINAL_STATUS"] = final_unique.apply(final_status, axis=1)

# ================================
# 16. ACTION
# ================================
action_map = {
    "Healthy": "No action",
    "Monitor": "Observe next period",
    "Possible Local Shift": "Review neighbor load / balancing",
    "Suspected Inactive": "Verify site and cell availability",
    "Area-wide Decline": "Check LAC / regional network issue",
    "High Value Degradation": "Immediate optimization required",
    "Local Area Degradation": "Investigate local cluster performance"
}

final_unique["ACTION"] = final_unique["FINAL_STATUS"].map(action_map)

# ================================
# 17. BUSINESS IMPACT
# ================================
def impact_tier(x):
    if x > 2000000:
        return "High"
    elif x > 500000:
        return "Medium"
    else:
        return "Low"

final_unique["BUSINESS_IMPACT"] = final_unique["IMPACT_SCORE"].apply(impact_tier)

# ================================
# 18. PRIORITY LABEL
# ================================
priority_map = {
    "CRITICAL": "P1 - Critical",
    "HIGH": "P2 - High",
    "MEDIUM": "P3 - Medium",
    "LOW": "P4 - Low"
}

final_unique["PRIORITY_LABEL"] = final_unique["PRIORITY"].map(priority_map)

priority_order = {"CRITICAL": 1, "HIGH": 2, "MEDIUM": 3, "LOW": 4}
final_unique["PRIORITY_RANK"] = final_unique["PRIORITY"].map(priority_order)

# ================================
# 19. SORT
# ================================
final_unique = final_unique.sort_values(
    by=["PRIORITY_RANK", "IMPACT_SCORE", "CONFIDENCE_SCORE"],
    ascending=[True, False, False]
)

# ================================
# 20. EXPORT
# ================================
final_unique.to_csv("FINAL_TELECOM_DASHBOARD_V4.csv", index=False)

# ================================
# 21. SUMMARY
# ================================
print("\n📊 TOTAL UNIQUE CELLS:", final_unique["CELL_ID"].nunique())
print("\n📊 STATUS COUNT:")
print(final_unique["FINAL_STATUS"].value_counts())
print("\n🔥 TOP 10 ISSUES:")
print(final_unique.head(10))

In [ ]:
final_unique.to_csv("FINAL_TELECOM_DASHBOARD_V4.csv", index=False)

In [ ]:
import os
os.getcwd()

In [ ]:
import pandas as pd
import numpy as np
from sklearn.neighbors import NearestNeighbors

# ================================
# 1. LOAD DATA
# ================================
file_path = r"C:\Users\malik\Downloads\LAC_CELL_CIRCLE_2G_SITES_DATA_FINAL.xlsx"

df1 = pd.read_excel(file_path, sheet_name=0)
df2 = pd.read_excel(file_path, sheet_name=1)
df3 = pd.read_excel(file_path, sheet_name=2)

df1["PERIOD"] = "May 8_10"
df2["PERIOD"] = "May 15_17"
df3["PERIOD"] = "May 22_24"

master_df = pd.concat([df1, df2, df3], ignore_index=True)

# ================================
# 2. CLEAN
# ================================
master_df.columns = (
    master_df.columns
    .str.strip()
    .str.upper()
    .str.replace(" ", "_")
)

# ================================
# 3. PERIOD FIX (LONG FORMAT)
# ================================
master_df["PERIOD_DATE"] = master_df["PERIOD"].str.extract(r'(\d+)').astype(int)

# Sort for rolling
master_df = master_df.sort_values(by=["CELL_ID", "PERIOD_DATE"])

# ================================
# 4. CORE METRICS (ROLLING)
# ================================
master_df["ROLLING_AVG"] = master_df.groupby("CELL_ID")["SESSION_COUNT"]\
    .transform(lambda x: x.shift(1).rolling(2, min_periods=1).mean())

master_df["ROLLING_AVG"] = master_df["ROLLING_AVG"].fillna(1)

master_df["PCT_CHANGE_ROLLING"] = (
    (master_df["SESSION_COUNT"] - master_df["ROLLING_AVG"]) /
    master_df["ROLLING_AVG"]
) * 100

master_df["LOSS"] = master_df["ROLLING_AVG"] - master_df["SESSION_COUNT"]

# ================================
# 5. TREND (3 PERIOD DROP)
# ================================
trend_df = master_df.groupby("CELL_ID").apply(
    lambda x: int(
        (len(x) >= 3) and
        (x.iloc[-1]["SESSION_COUNT"] < x.iloc[-2]["SESSION_COUNT"] < x.iloc[-3]["SESSION_COUNT"])
    )
).reset_index(name="TREND_DROP")

# ================================
# 6. LATEST PERIOD FILTER
# ================================
latest_df = master_df[master_df["PERIOD_DATE"] == 22].copy()
latest_df = latest_df.merge(trend_df, on="CELL_ID", how="left")

# ================================
# 7. LAC IMPACT
# ================================
lac_impact = latest_df.groupby("LAC_ID")["PCT_CHANGE_ROLLING"].mean().reset_index()
lac_impact.rename(columns={"PCT_CHANGE_ROLLING": "LAC_IMPACT"}, inplace=True)

latest_df = latest_df.merge(lac_impact, on="LAC_ID", how="left")

# ================================
# 8. LAT LONG (DUMMY)
# ================================
np.random.seed(42)
latest_df["LAT"] = np.random.uniform(28.40, 28.90, len(latest_df))
latest_df["LONG"] = np.random.uniform(76.90, 77.40, len(latest_df))

# ================================
# 9. NEIGHBOR ANALYSIS
# ================================
coords = latest_df[["LAT", "LONG"]]
nn = NearestNeighbors(n_neighbors=6)
nn.fit(coords)

distances, indices = nn.kneighbors(coords)

neighbor_data = []
for i in range(len(latest_df)):
    for j in range(1, 6):
        neighbor_data.append({
            "CELL_ID": latest_df.iloc[i]["CELL_ID"],
            "NEIGHBOR_CELL": latest_df.iloc[indices[i][j]]["CELL_ID"],
            "DISTANCE": distances[i][j]
        })

neighbor_df = pd.DataFrame(neighbor_data)

# ================================
# 10. NEIGHBOR GAIN
# ================================
neighbor_values = latest_df[["CELL_ID", "SESSION_COUNT"]].rename(
    columns={"CELL_ID": "NEIGHBOR_CELL", "SESSION_COUNT": "NEIGHBOR_CURRENT"}
)

cell_values = latest_df[["CELL_ID", "SESSION_COUNT"]].rename(
    columns={"SESSION_COUNT": "CELL_CURRENT"}
)

merged = neighbor_df.merge(cell_values, on="CELL_ID")
merged = merged.merge(neighbor_values, on="NEIGHBOR_CELL")

merged["NEIGHBOR_GAIN"] = merged["NEIGHBOR_CURRENT"] - merged["CELL_CURRENT"]
merged["POSITIVE_GAIN"] = np.where(merged["NEIGHBOR_GAIN"] > 0, merged["NEIGHBOR_GAIN"], 0)

neighbor_agg = merged.groupby("CELL_ID").agg({
    "POSITIVE_GAIN": "sum",
    "DISTANCE": "mean"
}).reset_index()

neighbor_agg.rename(columns={
    "POSITIVE_GAIN": "TOTAL_NEIGHBOR_GAIN",
    "DISTANCE": "AVG_NEIGHBOR_DISTANCE"
}, inplace=True)

final_df = latest_df.merge(neighbor_agg, on="CELL_ID", how="left")

# ================================
# 11. RECOVERY RATIO
# ================================
final_df["LOSS_SAFE"] = final_df["LOSS"].replace(0, 1)

final_df["RECOVERY_RATIO"] = (
    final_df["TOTAL_NEIGHBOR_GAIN"] / final_df["LOSS_SAFE"]
).clip(0, 2)

# ================================
# 12. IMPACT SCORE
# ================================
final_df["IMPACT_SCORE"] = (
    abs(final_df["PCT_CHANGE_ROLLING"]) * final_df["LOSS"]
)

# ================================
# 13. CONFIDENCE SCORE
# ================================
final_df["SESSION_SEVERITY"] = abs(final_df["PCT_CHANGE_ROLLING"]).clip(0, 100)
final_df["SIM_SEVERITY"] = (final_df["LOSS"] / final_df["LOSS"].max()) * 100
final_df["TREND_SCORE"] = final_df["TREND_DROP"] * 100
final_df["NEIGHBOR_SCORE"] = np.clip(final_df["RECOVERY_RATIO"] * 100, 0, 100)

trend_penalty = np.where(final_df["TREND_DROP"] == 0, -10, 0)

final_df["CONFIDENCE_SCORE"] = (
    0.35 * final_df["SIM_SEVERITY"] +
    0.30 * final_df["SESSION_SEVERITY"] +
    0.20 * final_df["TREND_SCORE"] +
    0.15 * final_df["NEIGHBOR_SCORE"] +
    trend_penalty
).clip(0, 100)

# ================================
# 14. PRIORITY
# ================================
def priority(row):
    if (row["IMPACT_SCORE"] > 1500000 and row["CONFIDENCE_SCORE"] >= 70):
        return "CRITICAL"
    elif (row["IMPACT_SCORE"] > 500000 and row["CONFIDENCE_SCORE"] >= 60):
        return "HIGH"
    elif (row["IMPACT_SCORE"] > 200000 and row["CONFIDENCE_SCORE"] >= 40):
        return "MEDIUM"
    else:
        return "LOW"

final_df["PRIORITY"] = final_df.apply(priority, axis=1)

# ================================
# 15. FINAL STATUS
# ================================
def final_status(row):
    if (row["PCT_CHANGE_ROLLING"] <= -85) and (row["CONFIDENCE_SCORE"] >= 70):
        return "Suspected Inactive"
    elif (row["PCT_CHANGE_ROLLING"] <= -70) and (row["RECOVERY_RATIO"] >= 0.5):
        return "Possible Local Shift"
    elif (row["LAC_IMPACT"] <= -40):
        return "Area-wide Decline"
    elif (row["PCT_CHANGE_ROLLING"] <= -50):
        return "Local Area Degradation"
    elif (row["PCT_CHANGE_ROLLING"] <= -40):
        return "Monitor"
    else:
        return "Healthy"

final_df["FINAL_STATUS"] = final_df.apply(final_status, axis=1)

# ================================
# 16. EXPORT
# ================================
final_df.to_csv("FINAL_TELECOM_DASHBOARD_V5.csv", index=False)

# ================================
# 17. SUMMARY CHECK
# ================================
print("\n📊 TOTAL UNIQUE CELLS:", final_df["CELL_ID"].nunique())
print("\n📊 STATUS COUNT:")
print(final_df["FINAL_STATUS"].value_counts())
print("\n🔥 TOP 10 ISSUES:")
print(final_df.sort_values(by=["IMPACT_SCORE"], ascending=False).head(10))

In [ ]:
import pandas as pd
import numpy as np
from sklearn.neighbors import NearestNeighbors

# ================================
# 1. LOAD DATA
# ================================
file_path = r"C:\Users\malik\Downloads\LAC_CELL_CIRCLE_2G_SITES_DATA_FINAL.xlsx"

df1 = pd.read_excel(file_path, sheet_name=0)
df2 = pd.read_excel(file_path, sheet_name=1)
df3 = pd.read_excel(file_path, sheet_name=2)

df1["PERIOD"] = "May 8_10"
df2["PERIOD"] = "May 15_17"
df3["PERIOD"] = "May 22_24"

master_df = pd.concat([df1, df2, df3], ignore_index=True)

# ================================
# 2. CLEAN
# ================================
master_df.columns = (
    master_df.columns
    .str.strip()
    .str.upper()
    .str.replace(" ", "_")
)

# ================================
# 3. PIVOT (SESSION + SIM BOTH)
# ================================
pivot_session = master_df.pivot_table(
    index=['CIRCLE_NAME', 'LAC_ID', 'CELL_ID'],
    columns='PERIOD',
    values='SESSION_COUNT',
    aggfunc='sum'
)

pivot_sim = master_df.pivot_table(
    index=['CIRCLE_NAME', 'LAC_ID', 'CELL_ID'],
    columns='PERIOD',
    values='SIM_COUNT',
    aggfunc='sum'
)

pivot_df = pivot_session.copy()
pivot_df.columns = [f"{col}_SESSION" for col in pivot_df.columns]

pivot_sim.columns = [f"{col}_SIM" for col in pivot_sim.columns]

pivot_df = pivot_df.join(pivot_sim).reset_index().fillna(0)

baseline = "May 8_10"
mid = "May 15_17"
current = "May 22_24"

# ================================
# 4. CORE METRICS (COMBINED TRAFFIC)
# ================================

# Combine SESSION + SIM (weighted)
pivot_df["BASELINE_TRAFFIC"] = (
    0.7 * pivot_df[f"{baseline}_SESSION"] +
    0.3 * pivot_df[f"{baseline}_SIM"]
)

pivot_df["MID_TRAFFIC"] = (
    0.7 * pivot_df[f"{mid}_SESSION"] +
    0.3 * pivot_df[f"{mid}_SIM"]
)

pivot_df["CURRENT_TRAFFIC"] = (
    0.7 * pivot_df[f"{current}_SESSION"] +
    0.3 * pivot_df[f"{current}_SIM"]
)

# Rolling avg
pivot_df["ROLLING_AVG"] = (
    (pivot_df["BASELINE_TRAFFIC"] + pivot_df["MID_TRAFFIC"]) / 2
).replace(0, 1)

# % drop
pivot_df["PCT_CHANGE_ROLLING"] = (
    (pivot_df["CURRENT_TRAFFIC"] - pivot_df["ROLLING_AVG"]) /
    pivot_df["ROLLING_AVG"]
) * 100

# Loss
pivot_df["LOSS"] = pivot_df["ROLLING_AVG"] - pivot_df["CURRENT_TRAFFIC"]

# ================================
# 6. LAC IMPACT (STRONGER)
# ================================

lac_stats = pivot_df.groupby("LAC_ID").agg({
    "PCT_CHANGE_ROLLING": "mean",
    "CELL_ID": "count"
}).reset_index()

lac_stats.rename(columns={
    "PCT_CHANGE_ROLLING": "LAC_IMPACT",
    "CELL_ID": "CELL_COUNT"
}, inplace=True)

pivot_df = pivot_df.merge(lac_stats, on="LAC_ID", how="left")

# ================================
# 7. LAT LONG (DUMMY)
# ================================
np.random.seed(42)
pivot_df["LAT"] = np.random.uniform(28.40, 28.90, len(pivot_df))
pivot_df["LONG"] = np.random.uniform(76.90, 77.40, len(pivot_df))

# ================================
# 8. NEIGHBOR ANALYSIS
# ================================
coords = pivot_df[["LAT", "LONG"]]
nn = NearestNeighbors(n_neighbors=6)
nn.fit(coords)

distances, indices = nn.kneighbors(coords)

neighbor_data = []
for i in range(len(pivot_df)):
    for j in range(1, 6):
        neighbor_data.append({
            "CELL_ID": pivot_df.iloc[i]["CELL_ID"],
            "NEIGHBOR_CELL": pivot_df.iloc[indices[i][j]]["CELL_ID"],
            "DISTANCE": distances[i][j]
        })

neighbor_df = pd.DataFrame(neighbor_data)

# ================================
# 9. NEIGHBOR GAIN
# ================================
neighbor_values = pivot_df[["CELL_ID", current]].rename(
    columns={"CELL_ID": "NEIGHBOR_CELL", current: "NEIGHBOR_CURRENT"}
)

cell_values = pivot_df[["CELL_ID", current]].rename(
    columns={current: "CELL_CURRENT"}
)

merged = neighbor_df.merge(cell_values, on="CELL_ID")
merged = merged.merge(neighbor_values, on="NEIGHBOR_CELL")

merged["NEIGHBOR_GAIN"] = merged["NEIGHBOR_CURRENT"] - merged["CELL_CURRENT"]
merged["POSITIVE_GAIN"] = np.where(merged["NEIGHBOR_GAIN"] > 0, merged["NEIGHBOR_GAIN"], 0)

neighbor_agg = merged.groupby("CELL_ID").agg({
    "POSITIVE_GAIN": "sum",
    "DISTANCE": "mean"
}).reset_index()

neighbor_agg.rename(columns={
    "POSITIVE_GAIN": "TOTAL_NEIGHBOR_GAIN",
    "DISTANCE": "AVG_NEIGHBOR_DISTANCE"
}, inplace=True)

final_table = pivot_df.merge(neighbor_agg, on="CELL_ID", how="left")

# ================================
# 10. RECOVERY RATIO
# ================================
final_table["LOSS_SAFE"] = final_table["LOSS"].replace(0, 1)

final_table["RECOVERY_RATIO"] = (
    final_table["TOTAL_NEIGHBOR_GAIN"] / final_table["LOSS_SAFE"]
).clip(0, 2)

# ================================
# 11. IMPACT SCORE (FIXED ORDER)
# ================================
final_table["IMPACT_SCORE"] = (
    abs(final_table["PCT_CHANGE_ROLLING"]) * final_table["LOSS"]
)

# ================================
# 12. UNIQUE CELL
# ================================
final_unique = final_table.drop_duplicates(subset=["CELL_ID"]).copy()

# ================================
# 13. CONFIDENCE
# ================================
final_unique["SESSION_SEVERITY"] = abs(final_unique["PCT_CHANGE_ROLLING"]).clip(0, 100)

final_unique["SIM_SEVERITY"] = (
    (final_unique["LOSS"] / final_unique["LOSS"].max()) * 100
)

final_unique["TREND_SCORE"] = final_unique["TREND_DROP"] * 100
final_unique["NEIGHBOR_SCORE"] = np.clip(final_unique["RECOVERY_RATIO"] * 100, 0, 100)

trend_penalty = np.where(final_unique["TREND_DROP"] == 0, -10, 0)
mismatch_penalty = np.where(
    abs(final_unique["SESSION_SEVERITY"] - final_unique["SIM_SEVERITY"]) > 40,
    -10, 0
)

final_unique["CONFIDENCE_SCORE"] = (
    0.30 * final_unique["SIM_SEVERITY"] +
    0.30 * final_unique["SESSION_SEVERITY"] +
    0.20 * final_unique["TREND_SCORE"] +
    0.20 * final_unique["NEIGHBOR_SCORE"] +
    trend_penalty + mismatch_penalty
).clip(0, 100)

def confidence_label(score):
    if score >= 80:
        return "High Confidence"
    elif score >= 60:
        return "Moderate"
    elif score >= 40:
        return "Watchlist"
    else:
        return "Low"

final_unique["CONFIDENCE_LEVEL"] = final_unique["CONFIDENCE_SCORE"].apply(confidence_label)

# ================================
# 14. PRIORITY (FIXED ✅)
# ================================
def priority(row):
    impact = row["IMPACT_SCORE"]
    confidence = row["CONFIDENCE_SCORE"]

    if (impact > 1500000 and confidence >= 70):
        return "CRITICAL"
    elif (impact > 500000 and confidence >= 60):
        return "HIGH"
    elif (impact > 200000 and confidence >= 40):
        return "MEDIUM"
    else:
        return "LOW"

final_unique["PRIORITY"] = final_unique.apply(priority, axis=1)

# ================================
# 15. FINAL STATUS
# ================================
def final_status(row):
    drop = row["PCT_CHANGE_ROLLING"]
    recovery = row["RECOVERY_RATIO"]
    lac = row["LAC_IMPACT"]
    trend = row["TREND_DROP"]
    loss = row["LOSS"]
    confidence = row["CONFIDENCE_SCORE"]
    current_val = row["CURRENT_TRAFFIC"]
    baseline_val = row["ROLLING_AVG"]

    # 🚨 Dead cell
    if (drop <= -85) and (current_val < 0.1 * baseline_val) and (confidence >= 70):
        return "Suspected Inactive"

    # 🔄 Traffic shift
    elif (drop <= -70) and (recovery >= 0.5):
        return "Possible Local Shift"

    # 🌍 Entire area down
    elif (lac <= -40) and (row["CELL_COUNT"] >= 5):
        return "Area-wide Decline"

    # 📉 Local cluster issue
    elif (lac <= -25) and (drop <= -40) and (recovery < 0.3):
        return "Local Area Degradation"

    # 💰 High business loss
    elif (loss > 40000) and (drop <= -50):
        return "High Value Degradation"

    # 👀 Watch
    elif (drop <= -40):
        return "Monitor"

    else:
        return "Healthy"

# ================================
# 16. ACTION
# ================================
action_map = {
    "Healthy": "No action",
    "Monitor": "Observe next period",
    "Possible Local Shift": "Review neighbor load / balancing",
    "Suspected Inactive": "Verify site and cell availability",
    "Area-wide Decline": "Check LAC / regional network issue",
    "High Value Degradation": "Immediate optimization required",
    "Local Area Degradation": "Investigate local cluster performance"
}

final_unique["ACTION"] = final_unique["FINAL_STATUS"].map(action_map)

# ================================
# 17. BUSINESS IMPACT
# ================================
def impact_tier(x):
    if x > 2000000:
        return "High"
    elif x > 500000:
        return "Medium"
    else:
        return "Low"

final_unique["BUSINESS_IMPACT"] = final_unique["IMPACT_SCORE"].apply(impact_tier)

# ================================
# 18. PRIORITY LABEL
# ================================
priority_map = {
    "CRITICAL": "P1 - Critical",
    "HIGH": "P2 - High",
    "MEDIUM": "P3 - Medium",
    "LOW": "P4 - Low"
}

final_unique["PRIORITY_LABEL"] = final_unique["PRIORITY"].map(priority_map)

priority_order = {"CRITICAL": 1, "HIGH": 2, "MEDIUM": 3, "LOW": 4}
final_unique["PRIORITY_RANK"] = final_unique["PRIORITY"].map(priority_order)

# ================================
# 19. SORT
# ================================
final_unique = final_unique.sort_values(
    by=["PRIORITY_RANK", "IMPACT_SCORE", "CONFIDENCE_SCORE"],
    ascending=[True, False, False]
)

# ================================
# 20. EXPORT
# ================================
final_unique.to_csv("FINAL_TELECOM_DASHBOARD_V4.csv", index=False)

# ================================
# 21. SUMMARY
# ================================
print("\n📊 TOTAL UNIQUE CELLS:", final_unique["CELL_ID"].nunique())
print("\n📊 STATUS COUNT:")
print(final_unique["FINAL_STATUS"].value_counts())
print("\n🔥 TOP 10 ISSUES:")
print(final_unique.head(10))

In [ ]:
import pandas as pd
import numpy as np
from sklearn.neighbors import NearestNeighbors

# ================================
# 1. LOAD DATA
# ================================
file_path = r"C:\Users\malik\Downloads\LAC_CELL_CIRCLE_2G_SITES_DATA_FINAL.xlsx"

df1 = pd.read_excel(file_path, sheet_name=0)
df2 = pd.read_excel(file_path, sheet_name=1)
df3 = pd.read_excel(file_path, sheet_name=2)

df1["PERIOD"] = "May 8_10"
df2["PERIOD"] = "May 15_17"
df3["PERIOD"] = "May 22_24"

master_df = pd.concat([df1, df2, df3], ignore_index=True)

# ================================
# 2. CLEAN
# ================================
master_df.columns = (
    master_df.columns
    .str.strip()
    .str.upper()
    .str.replace(" ", "_")
)

# ================================
# 3. PIVOT (SESSION + SIM)
# ================================
pivot_session = master_df.pivot_table(
    index=['CIRCLE_NAME', 'LAC_ID', 'CELL_ID'],
    columns='PERIOD',
    values='SESSION_COUNT',
    aggfunc='sum'
)

pivot_sim = master_df.pivot_table(
    index=['CIRCLE_NAME', 'LAC_ID', 'CELL_ID'],
    columns='PERIOD',
    values='SIM_COUNT',
    aggfunc='sum'
)

pivot_df = pivot_session.copy()
pivot_df.columns = [f"{col}_SESSION" for col in pivot_df.columns]

pivot_sim.columns = [f"{col}_SIM" for col in pivot_sim.columns]

pivot_df = pivot_df.join(pivot_sim).reset_index().fillna(0)

baseline = "May 8_10"
mid = "May 15_17"
current = "May 22_24"

# ================================
# 4. TRAFFIC (COMBINED)
# ================================
pivot_df["BASELINE_TRAFFIC"] = (
    0.7 * pivot_df[f"{baseline}_SESSION"] +
    0.3 * pivot_df[f"{baseline}_SIM"]
)

pivot_df["MID_TRAFFIC"] = (
    0.7 * pivot_df[f"{mid}_SESSION"] +
    0.3 * pivot_df[f"{mid}_SIM"]
)

pivot_df["CURRENT_TRAFFIC"] = (
    0.7 * pivot_df[f"{current}_SESSION"] +
    0.3 * pivot_df[f"{current}_SIM"]
)

# ================================
# 5. METRICS
# ================================
pivot_df["ROLLING_AVG"] = (
    (pivot_df["BASELINE_TRAFFIC"] + pivot_df["MID_TRAFFIC"]) / 2
).replace(0, 1)

pivot_df["PCT_CHANGE_ROLLING"] = (
    (pivot_df["CURRENT_TRAFFIC"] - pivot_df["ROLLING_AVG"]) /
    pivot_df["ROLLING_AVG"]
) * 100

pivot_df["LOSS"] = pivot_df["ROLLING_AVG"] - pivot_df["CURRENT_TRAFFIC"]

# ✅ TREND FIX
pivot_df["TREND_DROP"] = np.where(
    (pivot_df["CURRENT_TRAFFIC"] < pivot_df["MID_TRAFFIC"]) &
    (pivot_df["MID_TRAFFIC"] < pivot_df["BASELINE_TRAFFIC"]),
    1, 0
)

# ================================
# 6. LAC IMPACT
# ================================
lac_stats = pivot_df.groupby("LAC_ID").agg({
    "PCT_CHANGE_ROLLING": "mean",
    "CELL_ID": "count"
}).reset_index()

lac_stats.rename(columns={
    "PCT_CHANGE_ROLLING": "LAC_IMPACT",
    "CELL_ID": "CELL_COUNT"
}, inplace=True)

pivot_df = pivot_df.merge(lac_stats, on="LAC_ID", how="left")

# ================================
# 7. LAT LONG (DUMMY)
# ================================
np.random.seed(42)
pivot_df["LAT"] = np.random.uniform(28.40, 28.90, len(pivot_df))
pivot_df["LONG"] = np.random.uniform(76.90, 77.40, len(pivot_df))

# ================================
# 8. NEIGHBOR ANALYSIS
# ================================
coords = pivot_df[["LAT", "LONG"]]
nn = NearestNeighbors(n_neighbors=6)
nn.fit(coords)

distances, indices = nn.kneighbors(coords)

neighbor_data = []
for i in range(len(pivot_df)):
    for j in range(1, 6):
        neighbor_data.append({
            "CELL_ID": pivot_df.iloc[i]["CELL_ID"],
            "NEIGHBOR_CELL": pivot_df.iloc[indices[i][j]]["CELL_ID"],
            "DISTANCE": distances[i][j]
        })

neighbor_df = pd.DataFrame(neighbor_data)

# ================================
# 9. NEIGHBOR GAIN (FIXED)
# ================================
neighbor_values = pivot_df[["CELL_ID", "CURRENT_TRAFFIC"]].rename(
    columns={"CELL_ID": "NEIGHBOR_CELL", "CURRENT_TRAFFIC": "NEIGHBOR_CURRENT"}
)

cell_values = pivot_df[["CELL_ID", "CURRENT_TRAFFIC"]].rename(
    columns={"CURRENT_TRAFFIC": "CELL_CURRENT"}
)

merged = neighbor_df.merge(cell_values, on="CELL_ID")
merged = merged.merge(neighbor_values, on="NEIGHBOR_CELL")

merged["NEIGHBOR_GAIN"] = merged["NEIGHBOR_CURRENT"] - merged["CELL_CURRENT"]
merged["POSITIVE_GAIN"] = np.where(merged["NEIGHBOR_GAIN"] > 0, merged["NEIGHBOR_GAIN"], 0)

neighbor_agg = merged.groupby("CELL_ID").agg({
    "POSITIVE_GAIN": "sum",
    "DISTANCE": "mean"
}).reset_index()

neighbor_agg.rename(columns={
    "POSITIVE_GAIN": "TOTAL_NEIGHBOR_GAIN",
    "DISTANCE": "AVG_NEIGHBOR_DISTANCE"
}, inplace=True)

final_table = pivot_df.merge(neighbor_agg, on="CELL_ID", how="left")

# ================================
# 10. RECOVERY
# ================================
final_table["LOSS_SAFE"] = final_table["LOSS"].replace(0, 1)

final_table["RECOVERY_RATIO"] = (
    final_table["TOTAL_NEIGHBOR_GAIN"] / final_table["LOSS_SAFE"]
).clip(0, 2)

# ================================
# 11. IMPACT
# ================================
final_table["IMPACT_SCORE"] = (
    abs(final_table["PCT_CHANGE_ROLLING"]) * final_table["LOSS"]
)

# ================================
# 12. UNIQUE
# ================================
final_unique = final_table.drop_duplicates(subset=["CELL_ID"]).copy()

# ================================
# 13. CONFIDENCE
# ================================
final_unique["SESSION_SEVERITY"] = abs(final_unique["PCT_CHANGE_ROLLING"]).clip(0, 100)
final_unique["SIM_SEVERITY"] = (
    (final_unique["LOSS"] / final_unique["LOSS"].max()) * 100
)

final_unique["TREND_SCORE"] = final_unique["TREND_DROP"] * 100
final_unique["NEIGHBOR_SCORE"] = np.clip(final_unique["RECOVERY_RATIO"] * 100, 0, 100)

trend_penalty = np.where(final_unique["TREND_DROP"] == 0, -10, 0)
mismatch_penalty = np.where(
    abs(final_unique["SESSION_SEVERITY"] - final_unique["SIM_SEVERITY"]) > 40,
    -10, 0
)

final_unique["CONFIDENCE_SCORE"] = (
    0.30 * final_unique["SIM_SEVERITY"] +
    0.30 * final_unique["SESSION_SEVERITY"] +
    0.20 * final_unique["TREND_SCORE"] +
    0.20 * final_unique["NEIGHBOR_SCORE"] +
    trend_penalty + mismatch_penalty
).clip(0, 100)

# ================================
# 14. FINAL STATUS (FIXED APPLY)
# ================================
def final_status(row):
    drop = row["PCT_CHANGE_ROLLING"]
    recovery = row["RECOVERY_RATIO"]
    lac = row["LAC_IMPACT"]
    loss = row["LOSS"]
    confidence = row["CONFIDENCE_SCORE"]
    current_val = row["CURRENT_TRAFFIC"]
    baseline_val = row["ROLLING_AVG"]

    if (drop <= -85) and (current_val < 0.1 * baseline_val):
        return "Suspected Inactive"
    elif (drop <= -70) and (recovery >= 0.5):
        return "Possible Local Shift"
    elif (lac <= -40) and (row["CELL_COUNT"] >= 5):
        return "Area-wide Decline"
    elif (lac <= -25) and (drop <= -40):
        return "Local Area Degradation"
    elif (loss > 40000) and (drop <= -50):
        return "High Value Degradation"
    elif (drop <= -40):
        return "Monitor"
    else:
        return "Healthy"

final_unique["FINAL_STATUS"] = final_unique.apply(final_status, axis=1)

# ================================
# 15. EXPORT
# ================================
final_unique.to_csv("FINAL_TELECOM_DASHBOARD_V5.csv", index=False)

# ================================
# 16. SUMMARY
# ================================
print("\n📊 TOTAL UNIQUE CELLS:", final_unique["CELL_ID"].nunique())
print("\n📊 STATUS COUNT:")
print(final_unique["FINAL_STATUS"].value_counts())
print("\n🔥 TOP 10 ISSUES:")
print(final_unique.head(10))

In [ ]:
import pandas as pd
import numpy as np
from sklearn.neighbors import NearestNeighbors

# ================================
# 1. LOAD DATA
# ================================
file_path = r"C:\Users\malik\Downloads\LAC_CELL_CIRCLE_2G_SITES_DATA_FINAL.xlsx"

df1 = pd.read_excel(file_path, sheet_name=0)
df2 = pd.read_excel(file_path, sheet_name=1)
df3 = pd.read_excel(file_path, sheet_name=2)

df1["PERIOD"] = "May 8_10"
df2["PERIOD"] = "May 15_17"
df3["PERIOD"] = "May 22_24"

master_df = pd.concat([df1, df2, df3], ignore_index=True)

# ================================
# 2. CLEAN
# ================================
master_df.columns = (
    master_df.columns
    .str.strip()
    .str.upper()
    .str.replace(" ", "_")
)

# ================================
# 3. PIVOT
# ================================
pivot_df = master_df.pivot_table(
    index=['CIRCLE_NAME', 'LAC_ID', 'CELL_ID'],
    columns='PERIOD',
    values='SESSION_COUNT',
    aggfunc='sum'
).reset_index().fillna(0)

baseline = "May 8_10"
mid = "May 15_17"
current = "May 22_24"

# ================================
# 4. CORE METRICS
# ================================
pivot_df["ROLLING_AVG"] = pivot_df[[baseline, mid]].mean(axis=1).replace(0, 1)

pivot_df["PCT_CHANGE_ROLLING"] = (
    (pivot_df[current] - pivot_df["ROLLING_AVG"]) / pivot_df["ROLLING_AVG"]
) * 100

pivot_df["LOSS"] = pivot_df["ROLLING_AVG"] - pivot_df[current]

# ================================
# 5. TREND
# ================================
pivot_df["TREND_DROP"] = np.where(
    (pivot_df[current] < pivot_df[mid]) &
    (pivot_df[mid] < pivot_df[baseline]),
    1, 0
)

# ================================
# 6. LAC IMPACT
# ================================
lac_impact = pivot_df.groupby("LAC_ID")["PCT_CHANGE_ROLLING"].mean().reset_index()
lac_impact.rename(columns={"PCT_CHANGE_ROLLING": "LAC_IMPACT"}, inplace=True)
pivot_df = pivot_df.merge(lac_impact, on="LAC_ID", how="left")

# ================================
# 7. LAT LONG (DUMMY)
# ================================
np.random.seed(42)
pivot_df["LAT"] = np.random.uniform(28.40, 28.90, len(pivot_df))
pivot_df["LONG"] = np.random.uniform(76.90, 77.40, len(pivot_df))

# ================================
# 8. NEIGHBOR ANALYSIS
# ================================
coords = pivot_df[["LAT", "LONG"]]
nn = NearestNeighbors(n_neighbors=6)
nn.fit(coords)

distances, indices = nn.kneighbors(coords)

neighbor_data = []
for i in range(len(pivot_df)):
    for j in range(1, 6):
        neighbor_data.append({
            "CELL_ID": pivot_df.iloc[i]["CELL_ID"],
            "NEIGHBOR_CELL": pivot_df.iloc[indices[i][j]]["CELL_ID"],
            "DISTANCE": distances[i][j]
        })

neighbor_df = pd.DataFrame(neighbor_data)

# ================================
# 9. NEIGHBOR GAIN
# ================================
neighbor_values = pivot_df[["CELL_ID", current]].rename(
    columns={"CELL_ID": "NEIGHBOR_CELL", current: "NEIGHBOR_CURRENT"}
)

cell_values = pivot_df[["CELL_ID", current]].rename(
    columns={current: "CELL_CURRENT"}
)

merged = neighbor_df.merge(cell_values, on="CELL_ID")
merged = merged.merge(neighbor_values, on="NEIGHBOR_CELL")

merged["NEIGHBOR_GAIN"] = merged["NEIGHBOR_CURRENT"] - merged["CELL_CURRENT"]
merged["POSITIVE_GAIN"] = np.where(merged["NEIGHBOR_GAIN"] > 0, merged["NEIGHBOR_GAIN"], 0)

neighbor_agg = merged.groupby("CELL_ID").agg({
    "POSITIVE_GAIN": "sum",
    "DISTANCE": "mean"
}).reset_index()

neighbor_agg.rename(columns={
    "POSITIVE_GAIN": "TOTAL_NEIGHBOR_GAIN",
    "DISTANCE": "AVG_NEIGHBOR_DISTANCE"
}, inplace=True)

final_table = pivot_df.merge(neighbor_agg, on="CELL_ID", how="left")

# ================================
# 9. NEIGHBOR GAIN + DETAILS
# ================================
neighbor_values = pivot_df[["CELL_ID", current]].rename(
    columns={"CELL_ID": "NEIGHBOR_CELL", current: "NEIGHBOR_CURRENT"}
)

cell_values = pivot_df[["CELL_ID", current]].rename(
    columns={current: "CELL_CURRENT"}
)

merged = neighbor_df.merge(cell_values, on="CELL_ID")
merged = merged.merge(neighbor_values, on="NEIGHBOR_CELL")

merged["NEIGHBOR_GAIN"] = merged["NEIGHBOR_CURRENT"] - merged["CELL_CURRENT"]
merged["POSITIVE_GAIN"] = np.where(merged["NEIGHBOR_GAIN"] > 0, merged["NEIGHBOR_GAIN"], 0)

# 🔥 NEW: store neighbor list + traffic
neighbor_details = merged.groupby("CELL_ID").agg({
    "NEIGHBOR_CELL": lambda x: ", ".join(map(str, x)),
    "NEIGHBOR_CURRENT": lambda x: ", ".join(map(lambda v: str(int(v)), x))
}).reset_index()

neighbor_details.rename(columns={
    "NEIGHBOR_CELL": "TOP_NEIGHBORS",
    "NEIGHBOR_CURRENT": "TOP_NEIGHBOR_TRAFFIC"
}, inplace=True)

# existing aggregation
neighbor_agg = merged.groupby("CELL_ID").agg({
    "POSITIVE_GAIN": "sum",
    "DISTANCE": "mean"
}).reset_index()

neighbor_agg.rename(columns={
    "POSITIVE_GAIN": "TOTAL_NEIGHBOR_GAIN",
    "DISTANCE": "AVG_NEIGHBOR_DISTANCE"
}, inplace=True)

# merge all
final_table = pivot_df.merge(neighbor_agg, on="CELL_ID", how="left")
final_table = final_table.merge(neighbor_details, on="CELL_ID", how="left")

# ================================
# 10. RECOVERY RATIO
# ================================
final_table["LOSS_SAFE"] = final_table["LOSS"].replace(0, 1)

final_table["RECOVERY_RATIO"] = (
    final_table["TOTAL_NEIGHBOR_GAIN"] / final_table["LOSS_SAFE"]
).clip(0, 2)

# ================================
# 11. IMPACT SCORE (FIXED ORDER)
# ================================
final_table["IMPACT_SCORE"] = (
    abs(final_table["PCT_CHANGE_ROLLING"]) * final_table["LOSS"]
)

# ================================
# 12. UNIQUE CELL
# ================================
final_unique = final_table.drop_duplicates(subset=["CELL_ID"]).copy()

# ================================
# 13. CONFIDENCE
# ================================
final_unique["SESSION_SEVERITY"] = abs(final_unique["PCT_CHANGE_ROLLING"]).clip(0, 100)

final_unique["SIM_SEVERITY"] = (
    (final_unique["LOSS"] / final_unique["LOSS"].max()) * 100
)

final_unique["TREND_SCORE"] = final_unique["TREND_DROP"] * 100
final_unique["NEIGHBOR_SCORE"] = np.clip(final_unique["RECOVERY_RATIO"] * 100, 0, 100)

trend_penalty = np.where(final_unique["TREND_DROP"] == 0, -10, 0)
mismatch_penalty = np.where(
    abs(final_unique["SESSION_SEVERITY"] - final_unique["SIM_SEVERITY"]) > 40,
    -10, 0
)

final_unique["CONFIDENCE_SCORE"] = (
    0.30 * final_unique["SIM_SEVERITY"] +
    0.30 * final_unique["SESSION_SEVERITY"] +
    0.20 * final_unique["TREND_SCORE"] +
    0.20 * final_unique["NEIGHBOR_SCORE"] +
    trend_penalty + mismatch_penalty
).clip(0, 100)

def confidence_label(score):
    if score >= 80:
        return "High Confidence"
    elif score >= 60:
        return "Moderate"
    elif score >= 40:
        return "Watchlist"
    else:
        return "Low"

final_unique["CONFIDENCE_LEVEL"] = final_unique["CONFIDENCE_SCORE"].apply(confidence_label)

# ================================
# 14. PRIORITY (FIXED ✅)
# ================================
def priority(row):
    impact = row["IMPACT_SCORE"]
    confidence = row["CONFIDENCE_SCORE"]

    if (impact > 1500000 and confidence >= 70):
        return "CRITICAL"
    elif (impact > 500000 and confidence >= 60):
        return "HIGH"
    elif (impact > 200000 and confidence >= 40):
        return "MEDIUM"
    else:
        return "LOW"

final_unique["PRIORITY"] = final_unique.apply(priority, axis=1)

# ================================
# 15. FINAL STATUS
# ================================
def final_status(row):
    drop = row["PCT_CHANGE_ROLLING"]
    recovery = row["RECOVERY_RATIO"]
    lac = row["LAC_IMPACT"]
    trend = row["TREND_DROP"]
    loss = row["LOSS"]
    confidence = row["CONFIDENCE_SCORE"]
    current_val = row[current]
    baseline_val = row["ROLLING_AVG"]

    if (drop <= -85) and (current_val < 0.1 * baseline_val) and (confidence >= 70):
        return "Suspected Inactive"
    elif (drop <= -70) and (recovery >= 0.5) and (confidence >= 60):
        return "Possible Local Shift"
    elif (lac <= -40) and (recovery < 0.2):
        return "Area-wide Decline"
    elif (lac <= -25) and (drop <= -40):
        return "Local Area Degradation"
    elif (loss > 40000) and (drop <= -50):
        return "High Value Degradation"
    elif (drop <= -40):
        return "Monitor"
    else:
        return "Healthy"

final_unique["FINAL_STATUS"] = final_unique.apply(final_status, axis=1)

# ================================
# 16. ACTION
# ================================
action_map = {
    "Healthy": "No action",
    "Monitor": "Observe next period",
    "Possible Local Shift": "Review neighbor load / balancing",
    "Suspected Inactive": "Verify site and cell availability",
    "Area-wide Decline": "Check LAC / regional network issue",
    "High Value Degradation": "Immediate optimization required",
    "Local Area Degradation": "Investigate local cluster performance"
}

final_unique["ACTION"] = final_unique["FINAL_STATUS"].map(action_map)

# ================================
# 17. BUSINESS IMPACT
# ================================
def impact_tier(x):
    if x > 2000000:
        return "High"
    elif x > 500000:
        return "Medium"
    else:
        return "Low"

final_unique["BUSINESS_IMPACT"] = final_unique["IMPACT_SCORE"].apply(impact_tier)

# ================================
# 18. PRIORITY LABEL
# ================================
priority_map = {
    "CRITICAL": "P1 - Critical",
    "HIGH": "P2 - High",
    "MEDIUM": "P3 - Medium",
    "LOW": "P4 - Low"
}

final_unique["PRIORITY_LABEL"] = final_unique["PRIORITY"].map(priority_map)

priority_order = {"CRITICAL": 1, "HIGH": 2, "MEDIUM": 3, "LOW": 4}
final_unique["PRIORITY_RANK"] = final_unique["PRIORITY"].map(priority_order)

# ================================
# 19. SORT
# ================================
final_unique = final_unique.sort_values(
    by=["PRIORITY_RANK", "IMPACT_SCORE", "CONFIDENCE_SCORE"],
    ascending=[True, False, False]
)

# ================================
# 20. EXPORT
# ================================
final_unique.to_csv("FINAL_TELECOM_DASHBOARD_V4.csv", index=False)

# ================================
# 21. SUMMARY
# ================================
print("\n📊 TOTAL UNIQUE CELLS:", final_unique["CELL_ID"].nunique())
print("\n📊 STATUS COUNT:")
print(final_unique["FINAL_STATUS"].value_counts())
print("\n🔥 TOP 10 ISSUES:")
print(final_unique.head(10))

In [ ]:
print(final_unique.columns)

In [ ]:
print(final_table[[
    "CELL_ID",
    "TOP_NEIGHBORS",
    "TOP_NEIGHBOR_TRAFFIC"
]].head())

In [ ]:
df['date'] = pd.to_datetime(df['date'])

# filter 15–17 May
filtered = df[(df['date'] >= '2024-05-15') & (df['date'] <= '2024-05-17')]

# session > 10000
high_sessions = filtered[filtered['session_count'] > 10000]

# unique cells count
result = high_sessions['cell_id'].nunique()

print(result)

In [ ]:
import pandas as pd
import numpy as np
from sklearn.neighbors import NearestNeighbors

# ================================
# 1. LOAD DATA
# ================================
file_path = r"C:\Users\malik\Downloads\LAC_CELL_CIRCLE_2G_SITES_DATA_FINAL.xlsx"

df1 = pd.read_excel(file_path, sheet_name=0)
df2 = pd.read_excel(file_path, sheet_name=1)
df3 = pd.read_excel(file_path, sheet_name=2)

df1["PERIOD"] = "May 8_10"
df2["PERIOD"] = "May 15_17"
df3["PERIOD"] = "May 22_24"

master_df = pd.concat([df1, df2, df3], ignore_index=True)

# ================================
# 2. CLEAN
# ================================
master_df.columns = (
    master_df.columns
    .str.strip()
    .str.upper()
    .str.replace(" ", "_")
)

# ================================
# 2.5 CREATE LONG FORMAT (FOR TABLEAU)
# ================================
long_df = master_df.copy()

long_df.rename(columns={
    "SESSION_COUNT": "VALUE",
    "PERIOD": "DATE"
}, inplace=True)

# ================================
# 3. PIVOT
# ================================
pivot_df = master_df.pivot_table(
    index=['CIRCLE_NAME', 'LAC_ID', 'CELL_ID'],
    columns='PERIOD',
    values='SESSION_COUNT',
    aggfunc='sum'
).reset_index().fillna(0)
pivot_df = pivot_df.rename(columns={
    'May 8_10': 'May 8',
    'May 15_17': 'May 15',
    'May 22_24': 'May 22'
})

baseline = "May 8"
mid = "May 15"
current = "May 22"

# ================================
# 4. CORE METRICS
# ================================
pivot_df["ROLLING_AVG"] = pivot_df[[baseline, mid]].mean(axis=1).replace(0, 1)

pivot_df["PCT_CHANGE_ROLLING"] = (
    (pivot_df[current] - pivot_df["ROLLING_AVG"]) / pivot_df["ROLLING_AVG"]
) * 100

pivot_df["LOSS"] = pivot_df["ROLLING_AVG"] - pivot_df[current]

# ================================
# 5. TREND
# ================================
pivot_df["TREND_DROP"] = np.where(
    (pivot_df[current] < pivot_df[mid]) &
    (pivot_df[mid] < pivot_df[baseline]),
    1, 0
)

# ================================
# 6. LAC IMPACT
# ================================
lac_impact = pivot_df.groupby("LAC_ID")["PCT_CHANGE_ROLLING"].mean().reset_index()
lac_impact.rename(columns={"PCT_CHANGE_ROLLING": "LAC_IMPACT"}, inplace=True)
pivot_df = pivot_df.merge(lac_impact, on="LAC_ID", how="left")

# ================================
# 7. LAT LONG (DUMMY)
# ================================
np.random.seed(42)
pivot_df["LAT"] = np.random.uniform(28.40, 28.90, len(pivot_df))
pivot_df["LONG"] = np.random.uniform(76.90, 77.40, len(pivot_df))

# ================================
# 8. NEIGHBOR ANALYSIS
# ================================
coords = pivot_df[["LAT", "LONG"]]
nn = NearestNeighbors(n_neighbors=6)
nn.fit(coords)

distances, indices = nn.kneighbors(coords)

neighbor_data = []
for i in range(len(pivot_df)):
    for j in range(1, 6):
        neighbor_data.append({
            "CELL_ID": pivot_df.iloc[i]["CELL_ID"],
            "NEIGHBOR_CELL": pivot_df.iloc[indices[i][j]]["CELL_ID"],
            "DISTANCE": distances[i][j]
        })

neighbor_df = pd.DataFrame(neighbor_data)


# ================================
# 9. NEIGHBOR GAIN + DETAILS
# ================================
neighbor_values = pivot_df[["CELL_ID", current]].rename(
    columns={"CELL_ID": "NEIGHBOR_CELL", current: "NEIGHBOR_CURRENT"}
)

cell_values = pivot_df[["CELL_ID", current]].rename(
    columns={current: "CELL_CURRENT"}
)

merged = neighbor_df.merge(cell_values, on="CELL_ID")
merged = merged.merge(neighbor_values, on="NEIGHBOR_CELL")

merged["NEIGHBOR_GAIN"] = merged["NEIGHBOR_CURRENT"] - merged["CELL_CURRENT"]
merged["POSITIVE_GAIN"] = np.where(merged["NEIGHBOR_GAIN"] > 0, merged["NEIGHBOR_GAIN"], 0)

# 🔥 NEW: store neighbor list + traffic
neighbor_details = merged.groupby("CELL_ID").agg({
    "NEIGHBOR_CELL": lambda x: ", ".join(map(str, x)),
    "NEIGHBOR_CURRENT": lambda x: ", ".join(map(lambda v: str(int(v)), x))
}).reset_index()

neighbor_details.rename(columns={
    "NEIGHBOR_CELL": "TOP_NEIGHBORS",
    "NEIGHBOR_CURRENT": "TOP_NEIGHBOR_TRAFFIC"
}, inplace=True)

# existing aggregation
neighbor_agg = merged.groupby("CELL_ID").agg({
    "POSITIVE_GAIN": "sum",
    "DISTANCE": "mean"
}).reset_index()

neighbor_agg.rename(columns={
    "POSITIVE_GAIN": "TOTAL_NEIGHBOR_GAIN",
    "DISTANCE": "AVG_NEIGHBOR_DISTANCE"
}, inplace=True)

# merge all
final_table = pivot_df.merge(neighbor_agg, on="CELL_ID", how="left")
final_table = final_table.merge(neighbor_details, on="CELL_ID", how="left")

# ================================
# 10. RECOVERY RATIO
# ================================
final_table["LOSS_SAFE"] = final_table["LOSS"].replace(0, 1)

final_table["RECOVERY_RATIO"] = (
    final_table["TOTAL_NEIGHBOR_GAIN"] / final_table["LOSS_SAFE"]
).clip(0, 2)

# ================================
# 11. IMPACT SCORE (FIXED ORDER)
# ================================
final_table["IMPACT_SCORE"] = (
    abs(final_table["PCT_CHANGE_ROLLING"]) * final_table["LOSS"]
)

# ================================
# 12. UNIQUE CELL
# ================================
final_unique = final_table.drop_duplicates(subset=["CELL_ID"]).copy()

# ================================
# 13. CONFIDENCE
# ================================
final_unique["SESSION_SEVERITY"] = abs(final_unique["PCT_CHANGE_ROLLING"]).clip(0, 100)

final_unique["SIM_SEVERITY"] = (
    (final_unique["LOSS"] / final_unique["LOSS"].max()) * 100
)

final_unique["TREND_SCORE"] = final_unique["TREND_DROP"] * 100
final_unique["NEIGHBOR_SCORE"] = np.clip(final_unique["RECOVERY_RATIO"] * 100, 0, 100)

trend_penalty = np.where(final_unique["TREND_DROP"] == 0, -10, 0)
mismatch_penalty = np.where(
    abs(final_unique["SESSION_SEVERITY"] - final_unique["SIM_SEVERITY"]) > 40,
    -10, 0
)

final_unique["CONFIDENCE_SCORE"] = (
    0.30 * final_unique["SIM_SEVERITY"] +
    0.30 * final_unique["SESSION_SEVERITY"] +
    0.20 * final_unique["TREND_SCORE"] +
    0.20 * final_unique["NEIGHBOR_SCORE"] +
    trend_penalty + mismatch_penalty
).clip(0, 100)

def confidence_label(score):
    if score >= 80:
        return "High Confidence"
    elif score >= 60:
        return "Moderate"
    elif score >= 40:
        return "Watchlist"
    else:
        return "Low"

final_unique["CONFIDENCE_LEVEL"] = final_unique["CONFIDENCE_SCORE"].apply(confidence_label)

# ================================
# 14. PRIORITY (FIXED ✅)
# ================================
def priority(row):
    impact = row["IMPACT_SCORE"]
    confidence = row["CONFIDENCE_SCORE"]

    if (impact > 1500000 and confidence >= 70):
        return "CRITICAL"
    elif (impact > 500000 and confidence >= 60):
        return "HIGH"
    elif (impact > 200000 and confidence >= 40):
        return "MEDIUM"
    else:
        return "LOW"

final_unique["PRIORITY"] = final_unique.apply(priority, axis=1)

# ================================
# 15. FINAL STATUS
# ================================
def final_status(row):
    drop = row["PCT_CHANGE_ROLLING"]
    recovery = row["RECOVERY_RATIO"]
    lac = row["LAC_IMPACT"]
    trend = row["TREND_DROP"]
    loss = row["LOSS"]
    confidence = row["CONFIDENCE_SCORE"]
    current_val = row[current]
    baseline_val = row["ROLLING_AVG"]

    if (drop <= -85) and (current_val < 0.1 * baseline_val) and (confidence >= 70):
        return "Suspected Inactive"
    elif (drop <= -70) and (recovery >= 0.5) and (confidence >= 60):
        return "Possible Local Shift"
    elif (lac <= -40) and (recovery < 0.2):
        return "Area-wide Decline"
    elif (lac <= -25) and (drop <= -40):
        return "Local Area Degradation"
    elif (loss > 40000) and (drop <= -50):
        return "High Value Degradation"
    elif (drop <= -40):
        return "Monitor"
    else:
        return "Healthy"

final_unique["FINAL_STATUS"] = final_unique.apply(final_status, axis=1)

# ================================
# 16. ACTION
# ================================
action_map = {
    "Healthy": "No action",
    "Monitor": "Observe next period",
    "Possible Local Shift": "Review neighbor load / balancing",
    "Suspected Inactive": "Verify site and cell availability",
    "Area-wide Decline": "Check LAC / regional network issue",
    "High Value Degradation": "Immediate optimization required",
    "Local Area Degradation": "Investigate local cluster performance"
}

final_unique["ACTION"] = final_unique["FINAL_STATUS"].map(action_map)

# ================================
# 17. BUSINESS IMPACT
# ================================
def impact_tier(x):
    if x > 2000000:
        return "High"
    elif x > 500000:
        return "Medium"
    else:
        return "Low"

final_unique["BUSINESS_IMPACT"] = final_unique["IMPACT_SCORE"].apply(impact_tier)

# ================================
# 18. PRIORITY LABEL
# ================================
priority_map = {
    "CRITICAL": "P1 - Critical",
    "HIGH": "P2 - High",
    "MEDIUM": "P3 - Medium",
    "LOW": "P4 - Low"
}

final_unique["PRIORITY_LABEL"] = final_unique["PRIORITY"].map(priority_map)

priority_order = {"CRITICAL": 1, "HIGH": 2, "MEDIUM": 3, "LOW": 4}
final_unique["PRIORITY_RANK"] = final_unique["PRIORITY"].map(priority_order)

# ================================
# 19. SORT
# ================================
final_unique = final_unique.sort_values(
    by=["PRIORITY_RANK", "IMPACT_SCORE", "CONFIDENCE_SCORE"],
    ascending=[True, False, False]
)

# ================================
# FINAL EXPORT (FIXED ✅)
# ================================

# Step 1: Use pivot data (wide format)
long_df = pivot_df.copy()

# Step 2: Melt (so Tableau friendly bhi ho jaye)
long_df = long_df.melt(
    id_vars=["CIRCLE_NAME", "LAC_ID", "CELL_ID"],
    value_vars=["May 8", "May 15", "May 22"],
    var_name="DATE",
    value_name="VALUE"
)

# Step 3: Merge with final status
final_output = long_df.merge(
    final_unique,
    on=["CIRCLE_NAME", "LAC_ID", "CELL_ID"],
    how="left"
)

# ================================
# FINAL EXPORT (CLEAN ✅)
# ================================

# Step 1: Melt (Tableau friendly)
long_df = pivot_df.melt(
    id_vars=["CIRCLE_NAME", "LAC_ID", "CELL_ID"],
    value_vars=["May 8", "May 15", "May 22"],
    var_name="DATE",
    value_name="VALUE"
)

# Step 2: Proper full date mapping
date_map = {
    "May 8": "2026-05-08",
    "May 15": "2026-05-15",
    "May 22": "2026-05-22"
}

long_df["DATE"] = long_df["DATE"].map(date_map)
long_df["DATE"] = pd.to_datetime(long_df["DATE"])

# Step 3: Merge
final_output = long_df.merge(
    final_unique,
    on=["CIRCLE_NAME", "LAC_ID", "CELL_ID"],
    how="left"
)

# Export
final_output.to_csv("Airtel_Cell_Health_Analysis.csv", index=False)
# ================================
# 21. SUMMARY
# ================================
print("\n📊 TOTAL UNIQUE CELLS:", final_unique["CELL_ID"].nunique())
print("\n📊 STATUS COUNT:")
print(final_unique["FINAL_STATUS"].value_counts())
print("\n🔥 TOP 10 ISSUES:")
print(final_unique.head(10))

In [ ]:
final_output.to_csv("Airtel_Cell_Health_Analysis.csv", index=False)

In [ ]:
print(final_output.head())